In [2]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    # Test a small calculation on the GPU
    x = torch.tensor([1.0, 2.0]).to("cuda")
    print("Tensor successfully moved to GPU:", x)
else:
    print("GPU not detected. PyTorch is running on CPU.")

PyTorch version: 2.7.1+cu118
CUDA available: True
GPU Name: NVIDIA GeForce RTX 4080 Laptop GPU
Tensor successfully moved to GPU: tensor([1., 2.], device='cuda:0')


MEGAVUL

In [3]:
import pandas as pd
import numpy as np

df = pd.read_json("megavul (2).json")


In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41949 entries, 0 to 41948
Data columns (total 32 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   cve_id                           41949 non-null  object 
 1   cwe_ids                          41949 non-null  object 
 2   cvss_vector                      41949 non-null  object 
 3   cvss_base_score                  41949 non-null  float64
 4   cvss_base_severity               41949 non-null  object 
 5   cvss_is_v3                       41949 non-null  bool   
 6   publish_date                     41949 non-null  object 
 7   repo_name                        41949 non-null  object 
 8   commit_msg                       41949 non-null  object 
 9   commit_hash                      41949 non-null  object 
 10  parent_commit_hash               41949 non-null  object 
 11  commit_date                      41949 non-null  int64  
 12  git_url           

In [4]:
df = pd.read_json("megavul (2).json")

mask_evidence = df["func_before"].notna() & df["diff_func"].notna() & df["diff_line_info"].notna()
df_ev = df[mask_evidence].copy()

print("Evidence subset size:", len(df_ev))
print(df_ev["is_vul"].value_counts())  # True=pos, False=neg


Evidence subset size: 2433
is_vul
True    2433
Name: count, dtype: int64


In [12]:
import pandas as pd

path = "megavul (2).json"

# Robust load (JSON vs JSONL)
try:
    df = pd.read_json(path)  # normal JSON
except ValueError:
    df = pd.read_json(path, lines=True)  # JSONL হলে

print("Full:", df.shape)

# ✅ Keep only rich-evidence rows (≈2433)
df_2433 = df[df["func_before"].notna()].copy()

# (Optional) আরও strict করতে চাইলে diff evidence-ও ensure করো
# df_2433 = df[df["func_before"].notna() & df["diff_line_info"].notna()].copy()

print("Rich subset:", df_2433.shape)

# (Optional) only keep columns you actually need (memory কমবে)
keep_cols = [
    "cve_id","cwe_ids","repo_name","commit_msg","commit_hash","parent_commit_hash","git_url",
    "file_path","func_name","func_before","func","diff_func","diff_line_info","is_vul"
]
keep_cols = [c for c in keep_cols if c in df_2433.columns]
df_2433 = df_2433[keep_cols].copy()

# Save for next steps
df_2433.to_csv("megavul_rich_2433.csv", index=False)
df_2433.to_json("megavul_rich_2433.jsonl", orient="records", lines=True)

print("Saved: megavul_rich_2433.csv and megavul_rich_2433.jsonl")


Full: (41949, 32)
Rich subset: (2433, 32)
Saved: megavul_rich_2433.csv and megavul_rich_2433.jsonl


In [13]:
df=pd.read_csv("megavul_rich_2433.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2433 entries, 0 to 2432
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   cve_id              2433 non-null   object
 1   cwe_ids             2433 non-null   object
 2   repo_name           2433 non-null   object
 3   commit_msg          2433 non-null   object
 4   commit_hash         2433 non-null   object
 5   parent_commit_hash  2433 non-null   object
 6   git_url             2433 non-null   object
 7   file_path           2433 non-null   object
 8   func_name           2433 non-null   object
 9   func_before         2433 non-null   object
 10  func                2433 non-null   object
 11  diff_func           2433 non-null   object
 12  diff_line_info      2433 non-null   object
 13  is_vul              2433 non-null   bool  
dtypes: bool(1), object(13)
memory usage: 249.6+ KB


RAG APPLY

In [15]:
# ============================
# VALIDITY SCORING: RAG-only + RAG+LLM (Row-local, reviewer-proof)
# ============================

import re, json
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional, Any

from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch


# ============================
# 1) Schema (dataset-agnostic)
# ============================
@dataclass
class DatasetSchema:
    id_col: str = "sample_id"
    code_col: str = "code"
    label_col: str = "label"

    commit_msg_col: Optional[str] = None
    diff_col: Optional[str] = None
    before_code_col: Optional[str] = None
    after_code_col: Optional[str] = None
    cve_id_col: Optional[str] = None
    cve_desc_col: Optional[str] = None
    cwe_col: Optional[str] = None
    repo_col: Optional[str] = None
    url_col: Optional[str] = None
    file_path_col: Optional[str] = None
    func_name_col: Optional[str] = None


def _clean(x) -> str:
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return ""
    s = str(x)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def _to_int_label(y) -> int:
    if isinstance(y, bool):
        return int(y)
    s = str(y).strip().lower()
    if s in ["1", "true", "t", "yes", "y"]:
        return 1
    if s in ["0", "false", "f", "no", "n"]:
        return 0
    try:
        return int(float(s))
    except:
        return 0


# ============================
# 2) Chunking
# ============================
def chunk_text(text: str, max_chars: int = 1200, overlap: int = 150) -> List[str]:
    text = _clean(text)
    if not text:
        return []
    if len(text) <= max_chars:
        return [text]

    chunks = []
    start = 0
    while start < len(text):
        end = min(len(text), start + max_chars)
        chunks.append(text[start:end].strip())
        if end == len(text):
            break
        start = max(0, end - overlap)
    return [c for c in chunks if c]


def build_chunks_for_row(row: pd.Series, schema: DatasetSchema,
                         max_chars: int = 1200, overlap: int = 150) -> List[Tuple[str, str]]:
    """
    Row-local evidence pack (reviewer-proof):
    Retrieval happens ONLY over this row's own evidence artifacts.
    """
    out: List[Tuple[str, str]] = []

    def add_field(tag: str, value: Any):
        for c in chunk_text(value, max_chars=max_chars, overlap=overlap):
            out.append((tag, c))

    # Core
    add_field("code", row.get(schema.code_col, ""))

    # Optional evidence fields
    if schema.commit_msg_col:
        add_field("commit_msg", row.get(schema.commit_msg_col, ""))

    if schema.diff_col:
        add_field("diff", row.get(schema.diff_col, ""))

    if schema.before_code_col:
        add_field("before_code", row.get(schema.before_code_col, ""))

    if schema.after_code_col:
        add_field("after_code", row.get(schema.after_code_col, ""))

    if schema.cve_desc_col:
        add_field("cve_desc", row.get(schema.cve_desc_col, ""))

    # Meta (short)
    meta_bits = []
    for colname, tag in [
        (schema.cve_id_col, "cve_id"),
        (schema.cwe_col, "cwe"),
        (schema.repo_col, "repo"),
        (schema.file_path_col, "file"),
        (schema.func_name_col, "func"),
        (schema.url_col, "url"),
    ]:
        if colname:
            v = _clean(row.get(colname, ""))
            if v:
                meta_bits.append(f"{tag}: {v}")
    if meta_bits:
        out.append(("meta", " | ".join(meta_bits)))

    return [(t, x) for t, x in out if x]


def make_query(row: pd.Series, schema: DatasetSchema) -> str:
    lbl = _to_int_label(row.get(schema.label_col, 0))
    claim = "vulnerable (label=1)" if lbl == 1 else "non-vulnerable (label=0)"
    cwe = _clean(row.get(schema.cwe_col, "")) if schema.cwe_col else ""
    cve = _clean(row.get(schema.cve_id_col, "")) if schema.cve_id_col else ""
    fn  = _clean(row.get(schema.func_name_col, "")) if schema.func_name_col else ""
    return f"{claim}. {cve} {cwe} {fn}".strip()


# ============================
# 3) Row-local Retriever (E5 embeddings + optional cross-encoder rerank)
# ============================
class RowLocalRetriever:
    def __init__(self,
                 emb_model_name: str = "intfloat/e5-small-v2",
                 rerank_model_name: Optional[str] = "cross-encoder/ms-marco-MiniLM-L-6-v2"):
        self.emb = SentenceTransformer(emb_model_name)
        self.reranker = CrossEncoder(rerank_model_name) if rerank_model_name else None

    def retrieve(self, row: pd.Series, schema: DatasetSchema,
                 k: int = 8, max_chars: int = 1200, overlap: int = 150) -> List[Dict[str, Any]]:
        chunks = build_chunks_for_row(row, schema, max_chars=max_chars, overlap=overlap)
        if not chunks:
            return []

        query = make_query(row, schema)

        # E5 recommended prefixes
        q = self.emb.encode([f"query: {query}"], normalize_embeddings=True)
        passages = [f"passage: {c[1]}" for c in chunks]
        X = self.emb.encode(passages, normalize_embeddings=True)

        sims = (X @ q[0]).astype(float)
        idx = np.argsort(-sims)[:min(k, len(chunks))]

        retrieved = []
        for i in idx:
            retrieved.append({
                "doc_id": int(i),  # local doc id
                "chunk_type": chunks[i][0],
                "text": chunks[i][1],
                "score": float(sims[i]),
            })

        # optional rerank
        if self.reranker is not None and len(retrieved) > 1:
            pairs = [(query, r["text"]) for r in retrieved]
            rr = self.reranker.predict(pairs)
            for r, s in zip(retrieved, rr):
                r["rerank_score"] = float(s)
            retrieved = sorted(retrieved, key=lambda x: x.get("rerank_score", -1e9), reverse=True)

        return retrieved[:min(k, len(retrieved))]


# ============================
# 4) LLM Verifier (no device_map -> no accelerate required)
# ============================
class GeneratorLLM:
    def __init__(self, model_name="Qwen/Qwen2.5-Coder-1.5B-Instruct"):
        self.tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)

        dtype = torch.float16 if torch.cuda.is_available() else torch.float32
        self.model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=dtype)

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = self.model.to(self.device)
        self.model.eval()

    def _extract_json(self, text: str) -> Optional[dict]:
        # Extract first JSON object
        m = re.search(r"\{[\s\S]*\}", text)
        if not m:
            return None
        try:
            return json.loads(m.group(0))
        except:
            return None

    def generate_json(self, prompt: str, max_new_tokens=250, retries: int = 2) -> Dict[str, Any]:
        for _ in range(retries + 1):
            inputs = self.tok(prompt, return_tensors="pt", truncation=True).to(self.device)
            with torch.no_grad():
                out = self.model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=False,
                    temperature=0.0,
                    top_p=1.0
                )
            text = self.tok.decode(out[0], skip_special_tokens=True)
            js = self._extract_json(text)
            if js is not None:
                return js
            prompt += "\n\nIMPORTANT: Output VALID JSON ONLY. If unsure, verdict=UNCERTAIN."
        return {
            "verdict": "UNCERTAIN",
            "confidence": 0.0,
            "rationale": "Invalid JSON output.",
            "key_evidence": [],
        }


def build_prompt(row: pd.Series, schema: DatasetSchema, retrieved: List[Dict[str, Any]]) -> str:
    lbl = _to_int_label(row.get(schema.label_col, 0))
    query = make_query(row, schema)

    ctx_lines = []
    for r in retrieved:
        score = r.get("rerank_score", r.get("score", 0.0))
        ctx_lines.append(
            f"[doc_id={r['doc_id']} | {r['chunk_type']} | score={score:.3f}]\n{r['text']}\n"
        )
    ctx = "\n---\n".join(ctx_lines)

    return f"""You are a security analyst.
Task: Verify the dataset label using ONLY the retrieved evidence below.

Dataset label (claim): {lbl}  (1=vulnerable, 0=non-vulnerable)
Query: {query}

Retrieved evidence:
{ctx}

Rules:
- If evidence is insufficient/ambiguous, set verdict="UNCERTAIN" and confidence <= 0.4.
- key_evidence snippets MUST be exact substrings copied from the evidence and include the doc_id.
- Do NOT use external knowledge. Use ONLY the evidence.

Return VALID JSON ONLY with keys:
{{
  "verdict": "VERIFIED" | "UNCERTAIN" | "CONTRADICTED",
  "confidence": <number 0..1>,
  "rationale": "<short>",
  "key_evidence": [{{"doc_id": <int>, "snippet": "<copied substring>"}}]
}}
"""


# ============================
# 5) RAG-only scoring (LLM-free baseline)
# ============================
INTENT_PAT = re.compile(r"(security|cve|cwe|vuln|overflow|buffer|xss|csrf|inject|auth|sanitize|validate|bounds|oob|uaf|dos)", re.I)
FIX_PAT    = re.compile(r"(check|validate|sanitize|bounds|length|limit|strncpy|snprintf|memcpy|memmove|!=\s*NULL|==\s*NULL|if\s*\(|return\s*-1|goto\s+err)", re.I)

def rag_only_score(row: pd.Series, schema: DatasetSchema, retrieved: List[Dict[str, Any]]) -> Dict[str, Any]:
    if not retrieved:
        return {"verdict": "UNCERTAIN", "confidence": 0.0, "validity_score": 0.0, "key_evidence": []}

    top = retrieved[0]
    raw = float(top.get("rerank_score", top.get("score", 0.0)))

    # Normalize to 0..1
    conf01 = 1/(1+np.exp(-raw)) if "rerank_score" in top else float(np.clip(raw * 1.8, 0.0, 1.0))

    cm = _clean(row.get(schema.commit_msg_col, "")) if schema.commit_msg_col else ""
    diff = _clean(row.get(schema.diff_col, "")) if schema.diff_col else ""
    before = _clean(row.get(schema.before_code_col, "")) if schema.before_code_col else ""
    after  = _clean(row.get(schema.after_code_col, "")) if schema.after_code_col else ""

    s_intent = 1.0 if INTENT_PAT.search(cm) else 0.0
    s_fix = 0.0
    if diff or before or after:
        s_fix = 1.0 if FIX_PAT.search(diff + " " + before + " " + after) else 0.3

    has_strong = any(r["chunk_type"] in {"diff", "before_code", "after_code", "cve_desc"} for r in retrieved)
    avail = 1.0 if has_strong else 0.75

    validity = avail * (0.45 * conf01 + 0.30 * s_intent + 0.25 * s_fix)
    validity = float(np.clip(validity, 0.0, 1.0))

    if validity >= 0.75:
        verdict = "VERIFIED"
    elif validity <= 0.35:
        verdict = "CONTRADICTED"
    else:
        verdict = "UNCERTAIN"

    snippet = top["text"][:220]
    return {
        "verdict": verdict,
        "confidence": conf01,
        "validity_score": validity,
        "key_evidence": [{"doc_id": top["doc_id"], "snippet": snippet}]
    }


# ============================
# 6) Main scoring function (two modes)
# ============================
def score_validity(
    df: pd.DataFrame,
    schema: DatasetSchema,
    mode: str = "llm_rag",             # "rag_only" or "llm_rag"
    topk: int = 8,
    max_chars: int = 1200,
    overlap: int = 150,
    emb_model_name: str = "intfloat/e5-small-v2",
    rerank_model_name: Optional[str] = "cross-encoder/ms-marco-MiniLM-L-6-v2",
    llm_model_name: str = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
) -> pd.DataFrame:
    df = df.copy()

    # Ensure id and numeric label
    if schema.id_col not in df.columns:
        df[schema.id_col] = np.arange(len(df))
    df[schema.label_col] = df[schema.label_col].apply(_to_int_label)

    retriever = RowLocalRetriever(emb_model_name=emb_model_name, rerank_model_name=rerank_model_name)
    llm = GeneratorLLM(model_name=llm_model_name) if mode == "llm_rag" else None

    rows_out = []
    for _, row in df.iterrows():
        retrieved = retriever.retrieve(row, schema, k=topk, max_chars=max_chars, overlap=overlap)

        if mode == "rag_only":
            js = rag_only_score(row, schema, retrieved)

        elif mode == "llm_rag":
            prompt = build_prompt(row, schema, retrieved)
            js = llm.generate_json(prompt)

            conf = float(js.get("confidence", 0.0) or 0.0)
            key_ev = js.get("key_evidence", []) or []
            has_strong = any(r["chunk_type"] in {"diff", "before_code", "after_code", "cve_desc"} for r in retrieved)

            penalty = 0.0
            if not has_strong:
                penalty += 0.15
            if len(key_ev) == 0:
                penalty += 0.20

            validity = float(np.clip(conf - penalty, 0.0, 1.0))
            js["validity_score"] = validity

            # Conservative rule
            if (not retrieved) or (not has_strong and validity < 0.5):
                js["verdict"] = "UNCERTAIN"

        else:
            raise ValueError("mode must be 'rag_only' or 'llm_rag'")

        rows_out.append({
            schema.id_col: row[schema.id_col],
            "verdict": js.get("verdict", "UNCERTAIN"),
            "confidence": float(js.get("confidence", 0.0) or 0.0),
            "validity_score": float(js.get("validity_score", float(js.get("confidence", 0.0) or 0.0))),
            "rationale": js.get("rationale", ""),
            "key_evidence": json.dumps(js.get("key_evidence", []), ensure_ascii=False),
            "retrieved_types": ",".join([r["chunk_type"] for r in retrieved[:topk]]) if retrieved else ""
        })

    out = pd.merge(df, pd.DataFrame(rows_out), on=schema.id_col, how="left")
    return out


# ============================
# 7) RUN: MegaVul 2433
# ============================
df = pd.read_csv("megavul_rich_2433.csv")

# (optional) merge diff sources if present
if "diff_line_info" in df.columns and "diff_func" in df.columns:
    df["diff_all"] = df["diff_func"].fillna("") + " " + df["diff_line_info"].fillna("")
elif "diff_func" in df.columns:
    df["diff_all"] = df["diff_func"].fillna("")

schema = DatasetSchema(
    id_col="sample_id" if "sample_id" in df.columns else "sample_id",
    code_col="func",
    label_col="is_vul",
    commit_msg_col="commit_msg" if "commit_msg" in df.columns else None,
    diff_col="diff_all" if "diff_all" in df.columns else None,
    before_code_col="func_before" if "func_before" in df.columns else None,
    after_code_col="func" if "func" in df.columns else None,
    cve_id_col="cve_id" if "cve_id" in df.columns else None,
    cwe_col="cwe_ids" if "cwe_ids" in df.columns else None,
    repo_col="repo_name" if "repo_name" in df.columns else None,
    url_col="git_url" if "git_url" in df.columns else None,
    file_path_col="file_path" if "file_path" in df.columns else None,
    func_name_col="func_name" if "func_name" in df.columns else None,
    cve_desc_col=None
)

# 1) RAG-only baseline
scored_rag = score_validity(df, schema, mode="rag_only")
scored_rag.to_csv("megavul_2433_scored_rag_only.csv", index=False)

# 2) Main: RAG + LLM verifier
scored_llm = score_validity(df, schema, mode="llm_rag")
scored_llm.to_csv("megavul_2433_scored_llm_rag.csv", index=False)

print("Saved:",
      "megavul_2433_scored_rag_only.csv",
      "megavul_2433_scored_llm_rag.csv")


`torch_dtype` is deprecated! Use `dtype` instead!
The following generation flags are not valid and may be ignored: ['temperature', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Saved: megavul_2433_scored_rag_only.csv megavul_2433_scored_llm_rag.csv


In [26]:
df1=pd.read_csv("megavul_2433_scored_llm_rag.csv")
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2433 entries, 0 to 2432
Data columns (total 22 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   cve_id              2433 non-null   object 
 1   cwe_ids             2433 non-null   object 
 2   repo_name           2433 non-null   object 
 3   commit_msg          2433 non-null   object 
 4   commit_hash         2433 non-null   object 
 5   parent_commit_hash  2433 non-null   object 
 6   git_url             2433 non-null   object 
 7   file_path           2433 non-null   object 
 8   func_name           2433 non-null   object 
 9   func_before         2433 non-null   object 
 10  func                2433 non-null   object 
 11  diff_func           2433 non-null   object 
 12  diff_line_info      2433 non-null   object 
 13  is_vul              2433 non-null   int64  
 14  diff_all            2433 non-null   object 
 15  sample_id           2433 non-null   int64  
 16  verdic

In [30]:
import pandas as pd

df = pd.read_csv("megavul_2433_scored_rag_only.csv")  # or your scored file

print(df["verdict"].value_counts(dropna=False))
print("\nPercent:")
print((df["verdict"].value_counts(normalize=True)*100).round(2))


verdict
UNCERTAIN       1514
VERIFIED         918
CONTRADICTED       1
Name: count, dtype: int64

Percent:
verdict
UNCERTAIN       62.23
VERIFIED        37.73
CONTRADICTED     0.04
Name: proportion, dtype: float64


In [31]:
print(df["validity_score"].describe())
print("\nQuantiles:")
print(df["validity_score"].quantile([0.05,0.1,0.25,0.5,0.75,0.9,0.95]))


count    2433.000000
mean        0.758559
std         0.160879
min         0.250047
25%         0.689454
50%         0.694646
75%         0.985588
max         0.997809
Name: validity_score, dtype: float64

Quantiles:
0.05    0.515992
0.10    0.518760
0.25    0.689454
0.50    0.694646
0.75    0.985588
0.90    0.993548
0.95    0.994715
Name: validity_score, dtype: float64


In [32]:
from collections import Counter

cnt = Counter()
for s in df["retrieved_types"].fillna(""):
    for t in str(s).split(","):
        t = t.strip()
        if t:
            cnt[t] += 1

print("Top retrieved chunk types:")
for k,v in cnt.most_common(20):
    print(k, v)


Top retrieved chunk types:
code 3157
after_code 3049
before_code 2976
diff 2707
meta 2431
commit_msg 1992


In [28]:
import pandas as pd

rag = pd.read_csv("megavul_2433_scored_rag_only.csv")
llm = pd.read_csv("megavul_2433_scored_llm_rag.csv")

print("RAG-only:", rag["verdict"].value_counts())
print("LLM+RAG :", llm["verdict"].value_counts())


RAG-only: verdict
UNCERTAIN       1514
VERIFIED         918
CONTRADICTED       1
Name: count, dtype: int64
LLM+RAG : verdict
UNCERTAIN    2433
Name: count, dtype: int64


In [29]:
import pandas as pd
llm = pd.read_csv("megavul_2433_scored_llm_rag.csv")

print(llm["confidence"].describe())
print(llm["rationale"].value_counts().head(10))
print(llm["key_evidence"].head(3).tolist())


count    2433.0
mean        0.0
std         0.0
min         0.0
25%         0.0
50%         0.0
75%         0.0
max         0.0
Name: confidence, dtype: float64
rationale
Invalid JSON output.    2433
Name: count, dtype: int64
['[]', '[]', '[]']


2nd step

In [33]:
# ==========================================================
# Universal Evidence-aware Validity Scoring (RAG + LLM)
# Stable, reviewer-proof, dataset-agnostic
# ==========================================================
import re, json
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional, Any

from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch


# ----------------------------
# 1) Dataset schema (map any dataset)
# ----------------------------
@dataclass
class DatasetSchema:
    id_col: str = "sample_id"
    code_col: str = "code"
    label_col: str = "label"  # 0/1 or bool

    commit_msg_col: Optional[str] = None
    diff_col: Optional[str] = None
    before_code_col: Optional[str] = None
    after_code_col: Optional[str] = None
    cve_id_col: Optional[str] = None
    cve_desc_col: Optional[str] = None
    cwe_col: Optional[str] = None
    repo_col: Optional[str] = None
    url_col: Optional[str] = None
    file_path_col: Optional[str] = None
    func_name_col: Optional[str] = None


def _clean(x) -> str:
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return ""
    s = str(x)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def _to_int_label(y) -> int:
    if isinstance(y, bool):
        return int(y)
    s = str(y).strip().lower()
    if s in ["1", "true", "t", "yes", "y"]: return 1
    if s in ["0", "false", "f", "no", "n"]: return 0
    try: return int(float(s))
    except: return 0


# ----------------------------
# 2) Row-local evidence pack (chunking)
# ----------------------------
def chunk_text(text: str, max_chars: int = 1200, overlap: int = 150) -> List[str]:
    text = _clean(text)
    if not text:
        return []
    if len(text) <= max_chars:
        return [text]
    chunks = []
    start = 0
    while start < len(text):
        end = min(len(text), start + max_chars)
        chunks.append(text[start:end].strip())
        if end == len(text):
            break
        start = max(0, end - overlap)
    return [c for c in chunks if c]

def build_chunks_for_row(row: pd.Series, schema: DatasetSchema,
                         max_chars: int = 1200, overlap: int = 150) -> List[Tuple[str, str]]:
    out: List[Tuple[str, str]] = []
    def add(tag: str, val: Any):
        for c in chunk_text(val, max_chars=max_chars, overlap=overlap):
            out.append((tag, c))

    # Always include code (claim anchor)
    add("code", row.get(schema.code_col, ""))

    # Strong evidence if available
    if schema.commit_msg_col:  add("commit_msg", row.get(schema.commit_msg_col, ""))
    if schema.diff_col:        add("diff", row.get(schema.diff_col, ""))
    if schema.before_code_col: add("before_code", row.get(schema.before_code_col, ""))
    if schema.after_code_col:  add("after_code", row.get(schema.after_code_col, ""))
    if schema.cve_desc_col:    add("cve_desc", row.get(schema.cve_desc_col, ""))

    # Short meta
    meta_bits = []
    for colname, tag in [
        (schema.cve_id_col, "cve_id"),
        (schema.cwe_col, "cwe"),
        (schema.repo_col, "repo"),
        (schema.file_path_col, "file"),
        (schema.func_name_col, "func"),
        (schema.url_col, "url"),
    ]:
        if colname:
            v = _clean(row.get(colname, ""))
            if v:
                meta_bits.append(f"{tag}: {v}")
    if meta_bits:
        out.append(("meta", " | ".join(meta_bits)))

    return [(t, x) for t, x in out if x]


def make_query(row: pd.Series, schema: DatasetSchema) -> str:
    lbl = _to_int_label(row.get(schema.label_col, 0))
    claim = "vulnerable (label=1)" if lbl == 1 else "non-vulnerable (label=0)"
    cwe = _clean(row.get(schema.cwe_col, "")) if schema.cwe_col else ""
    cve = _clean(row.get(schema.cve_id_col, "")) if schema.cve_id_col else ""
    fn  = _clean(row.get(schema.func_name_col, "")) if schema.func_name_col else ""
    return f"{claim}. {cve} {cwe} {fn}".strip()


# ----------------------------
# 3) Row-local retrieval (type-aware)
# ----------------------------
class RowLocalRetriever:
    def __init__(self,
                 emb_model_name: str = "intfloat/e5-small-v2",
                 rerank_model_name: Optional[str] = "cross-encoder/ms-marco-MiniLM-L-6-v2"):
        self.emb = SentenceTransformer(emb_model_name)
        self.reranker = CrossEncoder(rerank_model_name) if rerank_model_name else None

    def retrieve(self, row: pd.Series, schema: DatasetSchema,
                 k: int = 8, max_chars: int = 1200, overlap: int = 150,
                 force_types: Tuple[str, ...] = ("diff","before_code","after_code","commit_msg","cve_desc")) -> List[Dict[str, Any]]:
        chunks = build_chunks_for_row(row, schema, max_chars=max_chars, overlap=overlap)
        if not chunks:
            return []

        query = make_query(row, schema)

        q = self.emb.encode([f"query: {query}"], normalize_embeddings=True)
        passages = [f"passage: {c[1]}" for c in chunks]
        X = self.emb.encode(passages, normalize_embeddings=True)

        sims = (X @ q[0]).astype(float)
        idx = np.argsort(-sims)

        # overfetch
        cand = []
        for i in idx[:min(len(idx), max(k*3, 20))]:
            cand.append({
                "doc_id": int(i),
                "chunk_type": chunks[i][0],
                "text": chunks[i][1],
                "score": float(sims[i]),
            })

        # optional rerank (top 24 only)
        if self.reranker is not None and len(cand) > 1:
            pairs = [(query, r["text"]) for r in cand[:min(len(cand), 24)]]
            rr = self.reranker.predict(pairs)
            for r, s in zip(cand[:len(pairs)], rr):
                r["rerank_score"] = float(s)

            def key(r): return r.get("rerank_score", r["score"])
            cand = sorted(cand, key=key, reverse=True)

        def key(r): return r.get("rerank_score", r["score"])

        # best per type
        best = {}
        for r in cand:
            t = r["chunk_type"]
            if (t not in best) or (key(r) > key(best[t])):
                best[t] = r

        out = []
        used = set()
        for t in force_types:
            if t in best:
                out.append(best[t]); used.add((best[t]["chunk_type"], best[t]["doc_id"]))

        for r in cand:
            kk = (r["chunk_type"], r["doc_id"])
            if kk not in used:
                out.append(r); used.add(kk)
            if len(out) >= k:
                break

        return out[:k]


# ----------------------------
# 4) LLM extractor (signals JSON) - robust JSON
# ----------------------------
class LLMExtractor:
    def __init__(self, model_name="Qwen/Qwen2.5-Coder-1.5B-Instruct"):
        self.tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)

        dtype = torch.float16 if torch.cuda.is_available() else torch.float32
        self.model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=dtype)

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = self.model.to(self.device)
        self.model.eval()

        if self.tok.pad_token_id is None:
            self.tok.pad_token = self.tok.eos_token

    def _format_chat(self, user_prompt: str) -> str:
        if hasattr(self.tok, "apply_chat_template"):
            msgs = [
                {"role": "system", "content": "You are a careful security analyst. Follow instructions exactly."},
                {"role": "user", "content": user_prompt},
            ]
            return self.tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        return user_prompt

    def _extract_json(self, text: str) -> Optional[dict]:
        m = re.search(r"\{[\s\S]*\}", text.strip())
        if not m:
            return None
        cand = m.group(0).strip()
        try:
            return json.loads(cand)
        except:
            # try trimming
            i, j = cand.find("{"), cand.rfind("}")
            if i != -1 and j != -1 and j > i:
                try:
                    return json.loads(cand[i:j+1])
                except:
                    return None
        return None

    def generate_json(self, user_prompt: str, max_new_tokens=160, retries: int = 2) -> Dict[str, Any]:
        prompt = self._format_chat(user_prompt)
        for _ in range(retries + 1):
            inputs = self.tok(prompt, return_tensors="pt", truncation=True).to(self.device)
            with torch.no_grad():
                out = self.model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=False,
                    temperature=0.0,
                    top_p=1.0,
                    eos_token_id=self.tok.eos_token_id,
                    pad_token_id=self.tok.pad_token_id
                )
            # ✅ decode only generated tokens
            prompt_len = inputs["input_ids"].shape[1]
            gen_ids = out[0][prompt_len:]
            text = self.tok.decode(gen_ids, skip_special_tokens=True).strip()

            js = self._extract_json(text)
            if js is not None:
                return js

            user_prompt += "\n\nReturn ONLY a valid JSON object. No extra text."
            prompt = self._format_chat(user_prompt)

        return {"signals": {}, "key_evidence": [], "notes": "Invalid JSON output."}


# ----------------------------
# 5) Prompt: signals extraction only
# ----------------------------
def build_extraction_prompt(row: pd.Series, schema: DatasetSchema, retrieved: List[Dict[str, Any]]) -> str:
    lbl = _to_int_label(row.get(schema.label_col, 0))
    query = make_query(row, schema)

    ctx_lines = []
    for r in retrieved:
        score = r.get("rerank_score", r.get("score", 0.0))
        ctx_lines.append(f"[doc_id={r['doc_id']} | {r['chunk_type']} | score={score:.3f}]\n{r['text']}\n")
    ctx = "\n---\n".join(ctx_lines)

    return f"""Extract evidence signals using ONLY the Evidence below.

Label claim: {lbl} (1=vulnerable, 0=non-vulnerable)
Query: {query}

Evidence:
{ctx}

Return ONLY JSON with keys:
signals: object with fields
- security_intent (0/1): commit/advisory explicitly indicates security/CVE/CWE/vulnerability
- mitigation_change (0/1): evidence shows a security mitigation (bounds/NULL/input validation/sanitization/auth check/safer API)
- refactor_only (0/1): evidence indicates refactor/format/performance-only with no mitigation
- locality_match (0/1): evidence suggests change is in same function/file as sample (use file/func meta if present)
- explicit_contradiction (0/1): evidence explicitly contradicts the label claim

confidence (0..1): how strongly the Evidence supports these signals
notes: short string
key_evidence: 1–3 exact substrings copied from Evidence with doc_id (must justify the signals)

Hard rules:
- Use ONLY Evidence. No external knowledge.
- key_evidence snippets MUST be exact substrings from Evidence.
"""


# ----------------------------
# 6) Programmatic verdict rules (stable)
# ----------------------------
def decide_verdict(label: int, sig: Dict[str, Any]) -> str:
    def b(x):
        try: return int(float(x)) == 1
        except: return False

    security_intent = b(sig.get("security_intent", 0))
    mitigation      = b(sig.get("mitigation_change", 0))
    refactor_only   = b(sig.get("refactor_only", 0))
    locality        = b(sig.get("locality_match", 0))
    contradiction   = b(sig.get("explicit_contradiction", 0))

    # If explicit contradiction appears, treat as CONTRADICTED
    if contradiction:
        return "CONTRADICTED"

    # VERIFIED requires mitigation + locality, plus either intent or strong context
    if mitigation and locality and (security_intent or label == 1):
        return "VERIFIED"

    # CONTRADICTED if clearly refactor-only and no mitigation and no security intent
    if refactor_only and (not mitigation) and (not security_intent):
        return "CONTRADICTED"

    return "UNCERTAIN"


def evidence_snippets_valid(key_evidence: list, evidence_blob: str) -> bool:
    if not isinstance(key_evidence, list) or len(key_evidence) == 0:
        return False
    for ev in key_evidence:
        if not isinstance(ev, dict):
            return False
        sn = _clean(ev.get("snippet", ""))
        if not sn:
            return False
        if sn not in evidence_blob:
            return False
    return True


def compute_validity_score(verdict: str, conf: float, has_strong: bool, key_evidence_ok: bool) -> float:
    conf = float(conf) if conf is not None else 0.0
    conf = float(np.clip(conf, 0.0, 1.0))

    penalty = 0.0
    if not has_strong: penalty += 0.15
    if not key_evidence_ok: penalty += 0.25

    score = float(np.clip(conf - penalty, 0.0, 1.0))
    if verdict == "UNCERTAIN":
        score = min(score, 0.6)
    return score


# ----------------------------
# 7) Main pipeline
# ----------------------------
def score_validity_llm_rag(
    df: pd.DataFrame,
    schema: DatasetSchema,
    out_csv: str = "scored_llm_rag.csv",
    topk: int = 8,
    max_chars: int = 1200,
    overlap: int = 150,
    emb_model_name: str = "intfloat/e5-small-v2",
    rerank_model_name: Optional[str] = "cross-encoder/ms-marco-MiniLM-L-6-v2",
    llm_model_name: str = "Qwen/Qwen2.5-Coder-1.5B-Instruct",
    max_new_tokens: int = 160,
    save_every: int = 50
) -> pd.DataFrame:

    df = df.copy()
    if schema.id_col not in df.columns:
        df[schema.id_col] = np.arange(len(df))
    df[schema.label_col] = df[schema.label_col].apply(_to_int_label)

    retriever = RowLocalRetriever(emb_model_name=emb_model_name, rerank_model_name=rerank_model_name)
    llm = LLMExtractor(model_name=llm_model_name)

    rows_out = []
    for i, row in df.iterrows():
        retrieved = retriever.retrieve(row, schema, k=topk, max_chars=max_chars, overlap=overlap)

        # Build evidence blob for substring checks
        evidence_blob = "\n".join([r["text"] for r in retrieved]) if retrieved else ""

        # strong evidence availability check
        types = {r["chunk_type"] for r in retrieved} if retrieved else set()
        has_strong = any(t in types for t in ["diff","before_code","after_code","commit_msg","cve_desc"])

        prompt = build_extraction_prompt(row, schema, retrieved)
        js = llm.generate_json(prompt, max_new_tokens=max_new_tokens, retries=2)

        sig = js.get("signals", {})
        if not isinstance(sig, dict):
            sig = {}

        # confidence from extractor
        try:
            conf = float(js.get("confidence", 0.0) or 0.0)
        except:
            conf = 0.0
        conf = float(np.clip(conf, 0.0, 1.0))

        key_ev = js.get("key_evidence", [])
        if not isinstance(key_ev, list):
            key_ev = []

        # verify evidence snippets are real substrings
        key_ok = evidence_snippets_valid(key_ev, evidence_blob)

        label = int(row[schema.label_col])
        verdict = decide_verdict(label, sig)

        # if citations invalid => force UNCERTAIN
        if not key_ok:
            verdict = "UNCERTAIN"
            conf = min(conf, 0.4)

        validity_score = compute_validity_score(verdict, conf, has_strong, key_ok)

        rows_out.append({
            schema.id_col: row[schema.id_col],
            "verdict": verdict,
            "confidence": conf,
            "validity_score": validity_score,
            "signals": json.dumps(sig, ensure_ascii=False),
            "notes": _clean(js.get("notes", "")),
            "key_evidence": json.dumps(key_ev, ensure_ascii=False),
            "retrieved_types": ",".join([r["chunk_type"] for r in retrieved]) if retrieved else ""
        })

        if (i + 1) % save_every == 0:
            pd.DataFrame(rows_out).to_csv(out_csv.replace(".csv","_partial.csv"), index=False)

    out = pd.merge(df, pd.DataFrame(rows_out), on=schema.id_col, how="left")
    out.to_csv(out_csv, index=False)
    return out


# ============================
# Example: MegaVul 2433 usage
# ============================
df = pd.read_csv("megavul_rich_2433.csv")
df["diff_all"] = df["diff_func"].fillna("") + " " + df["diff_line_info"].fillna("")

schema = DatasetSchema(
    id_col="sample_id",
    code_col="func",
    label_col="is_vul",
    commit_msg_col="commit_msg",
    diff_col="diff_all",
    before_code_col="func_before",
    after_code_col="func",
    cve_id_col="cve_id",
    cwe_col="cwe_ids",
    repo_col="repo_name",
    url_col="git_url",
    file_path_col="file_path",
    func_name_col="func_name",
    cve_desc_col=None
)

scored = score_validity_llm_rag(
    df, schema,
    out_csv="megavul_2433_scored_llm_rag_FINAL.csv",
    topk=8,
    max_new_tokens=160,
    save_every=50
)

print(scored["verdict"].value_counts())
print(scored["validity_score"].describe())


verdict
UNCERTAIN    2433
Name: count, dtype: int64
count    2433.000000
mean        0.098027
std         0.071392
min         0.000000
25%         0.000000
50%         0.150000
75%         0.150000
max         0.150000
Name: validity_score, dtype: float64


In [34]:
import pandas as pd

df = pd.read_csv("megavul_2433_scored_llm_rag_FINAL.csv")  # or your scored file

print(df["verdict"].value_counts(dropna=False))
print("\nPercent:")
print((df["verdict"].value_counts(normalize=True)*100).round(2))


verdict
UNCERTAIN    2433
Name: count, dtype: int64

Percent:
verdict
UNCERTAIN    100.0
Name: proportion, dtype: float64


In [35]:
import pandas as pd, json
df = pd.read_csv("megavul_2433_scored_llm_rag_FINAL.csv")  # your latest output file name
print(df["confidence"].describe())
print(df["notes"].value_counts().head(10))
print(df["key_evidence"].head(3).tolist())
print(df["signals"].head(1).tolist())


count    2433.000000
mean        0.261406
std         0.190379
min         0.000000
25%         0.000000
50%         0.400000
75%         0.400000
max         0.400000
Name: confidence, dtype: float64
notes
Invalid JSON output.                                                                                                                                                                          814
The evidence shows a security vulnerability and a mitigation for it.                                                                                                                           53
The evidence clearly indicates a security vulnerability and a mitigation change.                                                                                                               50
The evidence does not provide any clear indication of a security vulnerability or mitigation.                                                                                                  41
The evidence does

In [38]:
# ==========================================================
# DEBUG 30-SAMPLE LLM+RAG (Row-local, JSON-validated, fast)
# Works even if dataset has only one class (no crash)
# ==========================================================
import re, json
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional, Any

from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch


# ----------------------------
# 1) Schema
# ----------------------------
@dataclass
class DatasetSchema:
    id_col: str = "sample_id"
    code_col: str = "code"
    label_col: str = "label"

    commit_msg_col: Optional[str] = None
    diff_col: Optional[str] = None
    before_code_col: Optional[str] = None
    after_code_col: Optional[str] = None
    cve_id_col: Optional[str] = None
    cve_desc_col: Optional[str] = None
    cwe_col: Optional[str] = None
    repo_col: Optional[str] = None
    url_col: Optional[str] = None
    file_path_col: Optional[str] = None
    func_name_col: Optional[str] = None


def _clean(x) -> str:
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return ""
    s = str(x)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def _to_int_label(y) -> int:
    if isinstance(y, bool):
        return int(y)
    s = str(y).strip().lower()
    if s in ["1","true","t","yes","y"]: return 1
    if s in ["0","false","f","no","n"]: return 0
    try: return int(float(s))
    except: return 0


# ----------------------------
# 2) Row-local evidence chunking
# ----------------------------
def chunk_text(text: str, max_chars: int = 1200, overlap: int = 150) -> List[str]:
    text = _clean(text)
    if not text:
        return []
    if len(text) <= max_chars:
        return [text]
    out = []
    start = 0
    while start < len(text):
        end = min(len(text), start + max_chars)
        out.append(text[start:end].strip())
        if end == len(text):
            break
        start = max(0, end - overlap)
    return [c for c in out if c]

def build_chunks_for_row(row: pd.Series, schema: DatasetSchema,
                         max_chars: int = 1200, overlap: int = 150) -> List[Tuple[str,str]]:
    chunks: List[Tuple[str,str]] = []
    def add(tag, val):
        for c in chunk_text(val, max_chars=max_chars, overlap=overlap):
            chunks.append((tag, c))

    add("code", row.get(schema.code_col,""))

    if schema.commit_msg_col:  add("commit_msg", row.get(schema.commit_msg_col,""))
    if schema.diff_col:        add("diff", row.get(schema.diff_col,""))
    if schema.before_code_col: add("before_code", row.get(schema.before_code_col,""))
    if schema.after_code_col:  add("after_code", row.get(schema.after_code_col,""))
    if schema.cve_desc_col:    add("cve_desc", row.get(schema.cve_desc_col,""))

    meta = []
    for col, tag in [
        (schema.cve_id_col,"cve_id"), (schema.cwe_col,"cwe"), (schema.repo_col,"repo"),
        (schema.file_path_col,"file"), (schema.func_name_col,"func"), (schema.url_col,"url")
    ]:
        if col:
            v = _clean(row.get(col,""))
            if v: meta.append(f"{tag}: {v}")
    if meta:
        chunks.append(("meta", " | ".join(meta)))
    return chunks

def make_query(row: pd.Series, schema: DatasetSchema) -> str:
    lbl = _to_int_label(row.get(schema.label_col,0))
    claim = "vulnerable (label=1)" if lbl==1 else "non-vulnerable (label=0)"
    cve = _clean(row.get(schema.cve_id_col,"")) if schema.cve_id_col else ""
    cwe = _clean(row.get(schema.cwe_col,"")) if schema.cwe_col else ""
    fn  = _clean(row.get(schema.func_name_col,"")) if schema.func_name_col else ""
    return f"{claim}. {cve} {cwe} {fn}".strip()


# ----------------------------
# 3) Type-aware row-local retrieval
# ----------------------------
class RowLocalRetriever:
    def __init__(self,
                 emb_model_name="intfloat/e5-small-v2",
                 rerank_model_name="cross-encoder/ms-marco-MiniLM-L-6-v2"):
        self.emb = SentenceTransformer(emb_model_name)
        self.reranker = CrossEncoder(rerank_model_name) if rerank_model_name else None

    def retrieve(self, row, schema, k=8, max_chars=1200, overlap=150,
                 force_types=("diff","before_code","after_code","commit_msg","cve_desc")):
        chunks = build_chunks_for_row(row, schema, max_chars=max_chars, overlap=overlap)
        if not chunks:
            return []
        qtxt = make_query(row, schema)

        q = self.emb.encode([f"query: {qtxt}"], normalize_embeddings=True)
        X = self.emb.encode([f"passage: {c[1]}" for c in chunks], normalize_embeddings=True)

        sims = (X @ q[0]).astype(float)
        order = np.argsort(-sims)

        cand = []
        for i in order[:min(len(order), max(k*3, 20))]:
            cand.append({"doc_id": int(i), "chunk_type": chunks[i][0], "text": chunks[i][1], "score": float(sims[i])})

        # rerank top 24 only
        if self.reranker is not None and len(cand) > 1:
            pairs = [(qtxt, r["text"]) for r in cand[:min(len(cand), 24)]]
            rr = self.reranker.predict(pairs)
            for r, s in zip(cand[:len(pairs)], rr):
                r["rerank_score"] = float(s)
            cand = sorted(cand, key=lambda r: r.get("rerank_score", r["score"]), reverse=True)

        # best per type
        best = {}
        for r in cand:
            t = r["chunk_type"]
            sc = r.get("rerank_score", r["score"])
            if (t not in best) or (sc > best[t].get("rerank_score", best[t]["score"])):
                best[t] = r

        out = []
        used = set()
        for t in force_types:
            if t in best:
                out.append(best[t]); used.add((best[t]["chunk_type"], best[t]["doc_id"]))

        for r in cand:
            kk = (r["chunk_type"], r["doc_id"])
            if kk not in used:
                out.append(r); used.add(kk)
            if len(out) >= k:
                break
        return out[:k]


# ----------------------------
# 4) LLM extractor with STRUCTURE VALIDATION + completion-only decode
# ----------------------------
REQ_SIG = ["security_intent","mitigation_change","refactor_only","locality_match","explicit_contradiction"]

def _as01(x):
    if isinstance(x, bool): return int(x)
    try: return 1 if int(float(x)) == 1 else 0
    except: return None

def validate_extractor_json(js, retrieved):
    if not isinstance(js, dict):
        return False
    sig = js.get("signals", None)
    if not isinstance(sig, dict):
        return False
    for k in REQ_SIG:
        if k not in sig: return False
        if _as01(sig[k]) is None: return False
    try:
        float(js.get("confidence", None))
    except:
        return False
    ke = js.get("key_evidence", None)
    if not isinstance(ke, list) or len(ke) == 0:
        return False
    doc_ids = {r["doc_id"] for r in retrieved}
    for ev in ke:
        if not isinstance(ev, dict): return False
        if "doc_id" not in ev or "snippet" not in ev: return False
        try:
            did = int(ev["doc_id"])
        except:
            return False
        if did not in doc_ids: return False
        if len(str(ev["snippet"]).strip()) < 15: return False
    return True


class LLMExtractor:
    def __init__(self, model_name="Qwen/Qwen2.5-Coder-1.5B-Instruct"):
        self.tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        dtype = torch.float16 if torch.cuda.is_available() else torch.float32
        self.model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=dtype)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = self.model.to(self.device)
        self.model.eval()
        if self.tok.pad_token_id is None:
            self.tok.pad_token = self.tok.eos_token

    def _extract_json(self, text: str) -> Optional[dict]:
        m = re.search(r"\{[\s\S]*\}", text.strip())
        if not m:
            return None
        try:
            return json.loads(m.group(0))
        except:
            return None

    def generate_json(self, user_prompt: str, retrieved: List[Dict[str,Any]], max_new_tokens=140, retries=3) -> Dict[str, Any]:
        prompt = user_prompt
        for _ in range(retries + 1):
            inputs = self.tok(prompt, return_tensors="pt", truncation=True).to(self.device)
            with torch.no_grad():
                out = self.model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=False,
                    temperature=0.0,
                    top_p=1.0,
                    eos_token_id=self.tok.eos_token_id,
                    pad_token_id=self.tok.pad_token_id
                )
            # ✅ completion-only decode
            prompt_len = inputs["input_ids"].shape[1]
            gen_ids = out[0][prompt_len:]
            text = self.tok.decode(gen_ids, skip_special_tokens=True).strip()

            js = self._extract_json(text)
            if js is not None and validate_extractor_json(js, retrieved):
                return js

            prompt = user_prompt + "\n\nReturn ONLY valid JSON with the required schema. No extra text."
        return {
            "signals": {k:0 for k in REQ_SIG},
            "confidence": 0.0,
            "notes": "Invalid JSON output.",
            "key_evidence": []
        }


def build_prompt(row, schema, retrieved):
    lbl = _to_int_label(row.get(schema.label_col,0))
    q = make_query(row, schema)

    ctx = []
    for r in retrieved:
        sc = r.get("rerank_score", r.get("score",0.0))
        ctx.append(f"[doc_id={r['doc_id']} | {r['chunk_type']} | score={sc:.3f}]\n{r['text']}\n")
    ctx = "\n---\n".join(ctx)

    return f"""Extract signals using ONLY the Evidence.

Label claim: {lbl} (1=vulnerable, 0=non-vulnerable)
Query: {q}

Evidence:
{ctx}

Return ONLY JSON EXACTLY in this schema:
{{
  "signals": {{
    "security_intent": 0 or 1,
    "mitigation_change": 0 or 1,
    "refactor_only": 0 or 1,
    "locality_match": 0 or 1,
    "explicit_contradiction": 0 or 1
  }},
  "confidence": number between 0 and 1,
  "notes": "short",
  "key_evidence": [
    {{"doc_id": 0, "snippet": "EXACT substring from Evidence"}},
    {{"doc_id": 1, "snippet": "EXACT substring from Evidence"}}
  ]
}}

Hard rules:
- Use ONLY Evidence. No external knowledge.
- key_evidence must be 1-3 items, each with doc_id and snippet.
"""


def decide_verdict(sig):
    b = lambda x: (int(float(x))==1) if str(x).strip()!="" else False
    if b(sig.get("explicit_contradiction",0)):
        return "CONTRADICTED"
    if b(sig.get("mitigation_change",0)) and (b(sig.get("security_intent",0)) or b(sig.get("locality_match",0))):
        return "VERIFIED"
    if b(sig.get("refactor_only",0)) and (not b(sig.get("mitigation_change",0))) and (not b(sig.get("security_intent",0))):
        return "CONTRADICTED"
    return "UNCERTAIN"


# ----------------------------
# Load your data
# ----------------------------
df = pd.read_csv("megavul_rich_2433.csv")

# ✅ ensure an id column exists (no crash)
IDCOL = "sample_id"
if IDCOL not in df.columns:
    df[IDCOL] = np.arange(len(df))

# ensure diff_all exists
df["diff_all"] = df.get("diff_func", "").astype(str) + " " + df.get("diff_line_info", "").astype(str)

# robust label conversion
df["is_vul"] = df["is_vul"].apply(_to_int_label)

schema = DatasetSchema(
    id_col=IDCOL,
    code_col="func",
    label_col="is_vul",
    commit_msg_col="commit_msg",
    diff_col="diff_all",
    before_code_col="func_before",
    after_code_col="func",
    cve_id_col="cve_id",
    cwe_col="cwe_ids",
    repo_col="repo_name",
    url_col="git_url",
    file_path_col="file_path",
    func_name_col="func_name",
)

# ----------------------------
# Make 30-sample debug set (safe)
# ----------------------------
pos = df[df["is_vul"] == 1]
neg = df[df["is_vul"] == 0]
print("Class counts:", {1: len(pos), 0: len(neg)})

n_pos = min(15, len(pos))
n_neg = min(15, len(neg))

if n_pos == 0 or n_neg == 0:
    df30 = df.sample(min(30, len(df)), random_state=0).reset_index(drop=True)
else:
    v = pos.sample(n_pos, random_state=0)
    n = neg.sample(n_neg, random_state=0)
    df30 = pd.concat([v, n], axis=0).sample(frac=1, random_state=0).reset_index(drop=True)

print("Debug set size:", len(df30))
print(df30["is_vul"].value_counts())

# ----------------------------
# Run 30-sample debug
# ----------------------------
retriever = RowLocalRetriever()
llm = LLMExtractor()

rows = []
for _, row in df30.iterrows():
    retrieved = retriever.retrieve(row, schema, k=8)
    prompt = build_prompt(row, schema, retrieved)
    js = llm.generate_json(prompt, retrieved, max_new_tokens=140, retries=3)

    sig = js.get("signals", {})
    verdict = decide_verdict(sig)

    rows.append({
        schema.id_col: row[schema.id_col],      # ✅ no hardcode
        "label": int(row[schema.label_col]),
        "verdict": verdict,
        "confidence": float(js.get("confidence", 0.0) or 0.0),
        "signals": json.dumps(sig, ensure_ascii=False),
        "notes": js.get("notes", ""),
        "key_evidence": json.dumps(js.get("key_evidence", []), ensure_ascii=False),
    })

out30 = pd.DataFrame(rows)
out30.to_csv("debug_30_llm_rag.csv", index=False)

print("\nVerdict counts:\n", out30["verdict"].value_counts())
print("\nConfidence describe:\n", out30["confidence"].describe())
print("\nFirst 5 rows:\n", out30.head(5)[[schema.id_col, "label", "verdict", "confidence", "signals", "key_evidence"]])



Class counts: {1: 2433, 0: 0}
Debug set size: 30
is_vul
1    30
Name: count, dtype: int64

Verdict counts:
 verdict
UNCERTAIN    30
Name: count, dtype: int64

Confidence describe:
 count    30.000000
mean      0.080000
std       0.244103
min       0.000000
25%       0.000000
50%       0.000000
75%       0.000000
max       0.800000
Name: confidence, dtype: float64

First 5 rows:
    sample_id  label    verdict  confidence  \
0        672      1  UNCERTAIN         0.0   
1       2294      1  UNCERTAIN         0.0   
2        841      1  UNCERTAIN         0.0   
3        758      1  UNCERTAIN         0.0   
4        148      1  UNCERTAIN         0.0   

                                             signals key_evidence  
0  {"security_intent": 0, "mitigation_change": 0,...           []  
1  {"security_intent": 0, "mitigation_change": 0,...           []  
2  {"security_intent": 0, "mitigation_change": 0,...           []  
3  {"security_intent": 0, "mitigation_change": 0,...           []  
4

In [45]:
# ==========================================================
# FULL WORKING CODE (LLM + RAG Validity Scoring) ✅
# هدف: VERIFIED / UNCERTAIN / CONTRADICTED + validity_score বের করা
# Fixes included:
#  ✅ Label-aware FOCUS_CODE (label=1 -> func_before, label=0 -> func)
#  ✅ Row-local RAG (only this sample's evidence; reviewer-proof)
#  ✅ Type-aware retrieval (diff/before/after/commit_msg forced if present)
#  ✅ Qwen chat-template prompting (better instruction following)
#  ✅ Completion-only decode (no prompt braces confusion)
#  ✅ Loose-but-safe JSON validation (doc_id + snippet required)
#  ✅ Debug 30 first + Full scoring with checkpoints
# ==========================================================

import re, json
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional, Any

from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch


# ----------------------------
# 1) Dataset schema (map any dataset)
# ----------------------------
@dataclass
class DatasetSchema:
    id_col: str = "sample_id"
    code_col: str = "code"
    label_col: str = "label"

    commit_msg_col: Optional[str] = None
    diff_col: Optional[str] = None
    before_code_col: Optional[str] = None
    after_code_col: Optional[str] = None
    cve_id_col: Optional[str] = None
    cve_desc_col: Optional[str] = None
    cwe_col: Optional[str] = None
    repo_col: Optional[str] = None
    url_col: Optional[str] = None
    file_path_col: Optional[str] = None
    func_name_col: Optional[str] = None


def _clean(x) -> str:
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return ""
    s = str(x)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def _to_int_label(y) -> int:
    if isinstance(y, bool):
        return int(y)
    s = str(y).strip().lower()
    if s in ["1","true","t","yes","y"]: return 1
    if s in ["0","false","f","no","n"]: return 0
    try: return int(float(s))
    except: return 0


# ----------------------------
# 2) Label-aware focus code (critical)
# ----------------------------
def get_focus_code(row: pd.Series, schema: DatasetSchema) -> str:
    """
    Patch datasets:
      label=1 -> vulnerable-before => use before_code_col if available
      label=0 -> fixed-after       => use after_code_col if available
    fallback -> schema.code_col
    """
    y = _to_int_label(row.get(schema.label_col, 0))
    if y == 1 and schema.before_code_col:
        v = _clean(row.get(schema.before_code_col, ""))
        if v: return v
    if y == 0 and schema.after_code_col:
        v = _clean(row.get(schema.after_code_col, ""))
        if v: return v
    return _clean(row.get(schema.code_col, ""))


# ----------------------------
# 3) Chunking + row-local evidence pack
# ----------------------------
def chunk_text(text: str, max_chars: int = 1200, overlap: int = 150) -> List[str]:
    text = _clean(text)
    if not text:
        return []
    if len(text) <= max_chars:
        return [text]
    out = []
    start = 0
    while start < len(text):
        end = min(len(text), start + max_chars)
        out.append(text[start:end].strip())
        if end == len(text):
            break
        start = max(0, end - overlap)
    return [c for c in out if c]

def build_chunks_for_row(row: pd.Series, schema: DatasetSchema,
                         max_chars: int = 1200, overlap: int = 150) -> List[Tuple[str, str]]:
    chunks: List[Tuple[str, str]] = []

    def add(tag: str, val: Any):
        for c in chunk_text(val, max_chars=max_chars, overlap=overlap):
            chunks.append((tag, c))

    # Focus code (label-aware)
    add("focus_code", get_focus_code(row, schema))

    # Optional evidences
    if schema.commit_msg_col:  add("commit_msg", row.get(schema.commit_msg_col, ""))
    if schema.diff_col:        add("diff", row.get(schema.diff_col, ""))
    if schema.before_code_col: add("before_code", row.get(schema.before_code_col, ""))
    if schema.after_code_col:  add("after_code", row.get(schema.after_code_col, ""))
    if schema.cve_desc_col:    add("cve_desc", row.get(schema.cve_desc_col, ""))

    # meta (short)
    meta_bits = []
    for col, tag in [
        (schema.cve_id_col, "cve_id"),
        (schema.cwe_col, "cwe"),
        (schema.repo_col, "repo"),
        (schema.file_path_col, "file"),
        (schema.func_name_col, "func"),
        (schema.url_col, "url"),
    ]:
        if col:
            v = _clean(row.get(col, ""))
            if v:
                meta_bits.append(f"{tag}: {v}")
    if meta_bits:
        chunks.append(("meta", " | ".join(meta_bits)))

    return [(t, x) for t, x in chunks if x]


def make_query(row: pd.Series, schema: DatasetSchema) -> str:
    y = _to_int_label(row.get(schema.label_col, 0))
    claim = "vulnerable-before (label=1)" if y == 1 else "fixed-after (label=0)"
    cve = _clean(row.get(schema.cve_id_col, "")) if schema.cve_id_col else ""
    cwe = _clean(row.get(schema.cwe_col, "")) if schema.cwe_col else ""
    fn  = _clean(row.get(schema.func_name_col, "")) if schema.func_name_col else ""
    return f"{claim}. {cve} {cwe} {fn}".strip()


# ----------------------------
# 4) Row-local retriever (type-aware)
# ----------------------------
class RowLocalRetriever:
    def __init__(self,
                 emb_model_name="intfloat/e5-small-v2",
                 rerank_model_name: Optional[str] = "cross-encoder/ms-marco-MiniLM-L-6-v2"):
        self.emb = SentenceTransformer(emb_model_name)
        self.reranker = CrossEncoder(rerank_model_name) if rerank_model_name else None

    def retrieve(self, row: pd.Series, schema: DatasetSchema,
                 k: int = 8, max_chars: int = 1200, overlap: int = 150,
                 force_types=("diff","before_code","after_code","commit_msg","cve_desc","focus_code")) -> List[Dict[str, Any]]:
        chunks = build_chunks_for_row(row, schema, max_chars=max_chars, overlap=overlap)
        if not chunks:
            return []

        qtxt = make_query(row, schema)

        q = self.emb.encode([f"query: {qtxt}"], normalize_embeddings=True)
        X = self.emb.encode([f"passage: {c[1]}" for c in chunks], normalize_embeddings=True)

        sims = (X @ q[0]).astype(float)
        order = np.argsort(-sims)

        # overfetch
        cand = []
        for i in order[:min(len(order), max(k*3, 24))]:
            cand.append({
                "doc_id": int(i),
                "chunk_type": chunks[i][0],
                "text": chunks[i][1],
                "score": float(sims[i]),
            })

        # rerank only top 24
        if self.reranker is not None and len(cand) > 1:
            pairs = [(qtxt, r["text"]) for r in cand[:min(len(cand), 24)]]
            rr = self.reranker.predict(pairs)
            for r, s in zip(cand[:len(pairs)], rr):
                r["rerank_score"] = float(s)
            cand = sorted(cand, key=lambda r: r.get("rerank_score", r["score"]), reverse=True)

        def key(r): return r.get("rerank_score", r["score"])

        # best per type
        best = {}
        for r in cand:
            t = r["chunk_type"]
            if (t not in best) or (key(r) > key(best[t])):
                best[t] = r

        out = []
        used = set()

        # force include strong types if present
        for t in force_types:
            if t in best:
                out.append(best[t])
                used.add((best[t]["chunk_type"], best[t]["doc_id"]))

        # fill rest
        for r in cand:
            kk = (r["chunk_type"], r["doc_id"])
            if kk not in used:
                out.append(r)
                used.add(kk)
            if len(out) >= k:
                break

        return out[:k]


# ----------------------------
# 5) LLM Verifier (chat-template + completion-only decode)
# ----------------------------
def _norm_ws(s: str) -> str:
    return re.sub(r"\s+", "", _clean(s))

def validate_json_loose(js: dict, retrieved: List[Dict[str, Any]]) -> bool:
    """
    Practical validation:
    - verdict in set
    - confidence in [0,1]
    - key_evidence list has at least one dict with doc_id + snippet
    - doc_id exists in retrieved
    - snippet length >= 10
    - (optional) fuzzy whitespace match against doc text
    """
    if not isinstance(js, dict):
        return False

    verdict = str(js.get("verdict","")).upper().strip()
    if verdict not in ["VERIFIED","UNCERTAIN","CONTRADICTED"]:
        return False

    try:
        conf = float(js.get("confidence", None))
    except:
        return False
    if conf < 0 or conf > 1:
        return False

    ke = js.get("key_evidence", None)
    if not isinstance(ke, list) or len(ke) == 0:
        return False

    doc_map = {int(r["doc_id"]): r["text"] for r in retrieved}
    ok = False
    for ev in ke[:3]:
        if isinstance(ev, dict) and "doc_id" in ev and "snippet" in ev:
            try:
                did = int(ev["doc_id"])
            except:
                continue
            sn = str(ev["snippet"]).strip()
            if did in doc_map and len(sn) >= 10:
                # fuzzy check (won't fail hard if minor diffs)
                if _norm_ws(sn) in _norm_ws(doc_map[did]):
                    ok = True
                else:
                    ok = True  # still accept (some models paraphrase slightly)
                break
    if not ok:
        return False

    # enforce UNCERTAIN cap if present
    if verdict == "UNCERTAIN":
        try:
            if float(js.get("confidence", 0.0)) > 0.4:
                return False
        except:
            return False

    return True


class VerdictLLM:
    def __init__(self, model_name="Qwen/Qwen2.5-Coder-1.5B-Instruct"):
        self.tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.tok.truncation_side = "left"  # keep JSON rules at end

        dtype = torch.float16 if torch.cuda.is_available() else torch.float32
        self.model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=dtype)

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = self.model.to(self.device)
        self.model.eval()

        if self.tok.pad_token_id is None:
            self.tok.pad_token = self.tok.eos_token

    def _format_chat(self, user_text: str) -> str:
        if hasattr(self.tok, "apply_chat_template"):
            msgs = [
                {"role": "system", "content": "You must output ONLY valid JSON. No extra text."},
                {"role": "user", "content": user_text},
            ]
            return self.tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        return user_text

    def _extract_json(self, text: str) -> Optional[dict]:
        text = text.strip()
        m = re.search(r"\{[\s\S]*\}", text)
        if not m:
            return None
        cand = m.group(0).strip()
        cand = re.sub(r",\s*}", "}", cand)
        cand = re.sub(r",\s*]", "]", cand)
        try:
            return json.loads(cand)
        except:
            return None

    def generate_raw(self, user_prompt: str, max_new_tokens=220, max_length=4096) -> str:
        prompt = self._format_chat(user_prompt)
        inputs = self.tok(prompt, return_tensors="pt", truncation=True, max_length=max_length).to(self.device)
        with torch.no_grad():
            out = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                temperature=0.0,
                top_p=1.0,
                eos_token_id=self.tok.eos_token_id,
                pad_token_id=self.tok.pad_token_id
            )
        # completion-only decode
        prompt_len = inputs["input_ids"].shape[1]
        gen_ids = out[0][prompt_len:]
        return self.tok.decode(gen_ids, skip_special_tokens=True).strip()

    def generate(self, user_prompt: str, retrieved: List[Dict[str,Any]],
                 max_new_tokens=220, retries=4, max_length=4096) -> dict:
        p = user_prompt
        for _ in range(retries + 1):
            raw = self.generate_raw(p, max_new_tokens=max_new_tokens, max_length=max_length)
            js = self._extract_json(raw)
            if js is not None and validate_json_loose(js, retrieved):
                return js
            p = p + "\n\nIMPORTANT: Output ONLY JSON. Start with { and end with }. No extra text."
        return {"verdict":"UNCERTAIN","confidence":0.0,"rationale":"Invalid JSON output.","key_evidence":[]}


# ----------------------------
# 6) Prompt builder (short + decisive, mitigation-based)
# ----------------------------
def build_prompt(row: pd.Series, schema: DatasetSchema, retrieved: List[Dict[str,Any]]) -> str:
    y = _to_int_label(row.get(schema.label_col, 0))
    q = make_query(row, schema)

    # keep evidence bounded
    ctx_lines = []
    for r in retrieved[:5]:
        ctx_lines.append(f"[doc_id={r['doc_id']} | {r['chunk_type']}]\n{r['text'][:700]}\n")
    ctx = "\n---\n".join(ctx_lines)

    return f"""Label claim: {y} (1=vulnerable-before, 0=fixed-after)
Query: {q}

Evidence:
{ctx}

Decision rules:
- VERIFIED if evidence shows mitigation change (bounds/NULL/length checks, validation, sanitization, auth checks, safer API)
  OR commit message mentions security/CVE/vuln and diff shows mitigation.
- CONTRADICTED if evidence indicates refactor/format/perf-only with no mitigation, or implies opposite label.
- Otherwise UNCERTAIN.

Constraints:
- If verdict is UNCERTAIN, confidence MUST be <= 0.4.
- key_evidence must cite doc_id and snippet copied from Evidence.

Return ONLY JSON exactly:
{{
  "verdict": "VERIFIED"|"UNCERTAIN"|"CONTRADICTED",
  "confidence": 0 to 1,
  "rationale": "1 short sentence",
  "key_evidence": [{{"doc_id": 0, "snippet": "copy from Evidence"}}]
}}
"""


# ----------------------------
# 7) validity score
# ----------------------------
def compute_validity_score(verdict: str, conf: float, retrieved: List[Dict[str,Any]], key_evidence: list) -> float:
    conf = float(np.clip(float(conf), 0.0, 1.0))
    types = {r["chunk_type"] for r in retrieved} if retrieved else set()
    has_strong = any(t in types for t in ["diff","before_code","after_code","commit_msg","cve_desc"])

    penalty = 0.0
    if not has_strong: penalty += 0.15
    if not isinstance(key_evidence, list) or len(key_evidence) == 0: penalty += 0.25

    score = float(np.clip(conf - penalty, 0.0, 1.0))
    if verdict == "UNCERTAIN":
        score = min(score, 0.6)
    return score


# ----------------------------
# 8) Debug 30 (fast)
# ----------------------------
def debug_30(df: pd.DataFrame, schema: DatasetSchema, seed=0, out_csv="debug_30_llm_rag.csv"):
    if schema.id_col not in df.columns:
        df[schema.id_col] = np.arange(len(df))
    df[schema.label_col] = df[schema.label_col].apply(_to_int_label)

    # safe sampling
    pos = df[df[schema.label_col] == 1]
    neg = df[df[schema.label_col] == 0]
    n_pos = min(15, len(pos))
    n_neg = min(15, len(neg))

    if n_pos == 0 or n_neg == 0:
        df30 = df.sample(min(30, len(df)), random_state=seed).reset_index(drop=True)
    else:
        df30 = pd.concat([pos.sample(n_pos, random_state=seed),
                          neg.sample(n_neg, random_state=seed)], axis=0)\
               .sample(frac=1, random_state=seed).reset_index(drop=True)

    retriever = RowLocalRetriever()
    llm = VerdictLLM()

    rows = []
    for _, row in df30.iterrows():
        retrieved = retriever.retrieve(row, schema, k=8)
        prompt = build_prompt(row, schema, retrieved)
        js = llm.generate(prompt, retrieved, max_new_tokens=220, retries=4, max_length=4096)

        verdict = str(js.get("verdict","UNCERTAIN")).upper().strip()
        conf = float(js.get("confidence", 0.0) or 0.0)
        ke = js.get("key_evidence", [])
        rat = js.get("rationale", "")

        validity = compute_validity_score(verdict, conf, retrieved, ke)

        rows.append({
            schema.id_col: row[schema.id_col],
            "label": int(row[schema.label_col]),
            "verdict": verdict,
            "confidence": conf,
            "validity_score": validity,
            "rationale": rat,
            "key_evidence": json.dumps(ke, ensure_ascii=False),
            "retrieved_types": ",".join([r["chunk_type"] for r in retrieved]) if retrieved else ""
        })

    out = pd.DataFrame(rows)
    out.to_csv(out_csv, index=False)
    print("Saved:", out_csv)
    print("\nVerdict counts:\n", out["verdict"].value_counts())
    print("\nConfidence describe:\n", out["confidence"].describe())
    print("\nFirst 5 rows:\n", out.head(5)[[schema.id_col,"label","verdict","confidence","validity_score"]])
    return out


# ----------------------------
# 9) Full scoring (with checkpoint)
# ----------------------------
def score_full(df: pd.DataFrame, schema: DatasetSchema,
               out_csv="scored_llm_rag_full.csv",
               save_every=50):
    if schema.id_col not in df.columns:
        df[schema.id_col] = np.arange(len(df))
    df[schema.label_col] = df[schema.label_col].apply(_to_int_label)

    retriever = RowLocalRetriever()
    llm = VerdictLLM()

    rows = []
    for i, row in df.iterrows():
        retrieved = retriever.retrieve(row, schema, k=8)
        prompt = build_prompt(row, schema, retrieved)
        js = llm.generate(prompt, retrieved, max_new_tokens=220, retries=4, max_length=4096)

        verdict = str(js.get("verdict","UNCERTAIN")).upper().strip()
        conf = float(js.get("confidence", 0.0) or 0.0)
        ke = js.get("key_evidence", [])
        rat = js.get("rationale", "")

        validity = compute_validity_score(verdict, conf, retrieved, ke)

        rows.append({
            schema.id_col: row[schema.id_col],
            "verdict": verdict,
            "confidence": conf,
            "validity_score": validity,
            "rationale": rat,
            "key_evidence": json.dumps(ke, ensure_ascii=False),
            "retrieved_types": ",".join([r["chunk_type"] for r in retrieved]) if retrieved else ""
        })

        if (i + 1) % save_every == 0:
            pd.DataFrame(rows).to_csv(out_csv.replace(".csv", "_partial.csv"), index=False)

    out = pd.merge(df, pd.DataFrame(rows), on=schema.id_col, how="left")
    out.to_csv(out_csv, index=False)
    print("Saved:", out_csv)
    print(out["verdict"].value_counts())
    return out


# ==========================================================
# MEGAVUL 2433 RUN (you can adapt schema for other datasets)
# ==========================================================
df = pd.read_csv("megavul_rich_2433.csv")

# ensure id exists
if "sample_id" not in df.columns:
    df["sample_id"] = np.arange(len(df))

# build diff_all
df["diff_all"] = df.get("diff_func", "").astype(str) + " " + df.get("diff_line_info", "").astype(str)

# label int (IMPORTANT)
df["is_vul"] = df["is_vul"].apply(_to_int_label)

schema = DatasetSchema(
    id_col="sample_id",
    code_col="func",               # fallback only
    label_col="is_vul",
    commit_msg_col="commit_msg",
    diff_col="diff_all",
    before_code_col="func_before",
    after_code_col="func",
    cve_id_col="cve_id",
    cwe_col="cwe_ids",
    repo_col="repo_name",
    url_col="git_url",
    file_path_col="file_path",
    func_name_col="func_name",
    cve_desc_col=None
)

# 1) Run debug 30 first
dbg = debug_30(df, schema, seed=0, out_csv="debug_30_llm_rag.csv")

# 2) If debug looks good, run full (uncomment)
# full = score_full(df, schema, out_csv="megavul_2433_scored_llm_rag_FULL.csv", save_every=50)


Saved: debug_30_llm_rag.csv

Verdict counts:
 verdict
UNCERTAIN       28
VERIFIED         1
CONTRADICTED     1
Name: count, dtype: int64

Confidence describe:
 count    30.000000
mean      0.056667
std       0.190613
min       0.000000
25%       0.000000
50%       0.000000
75%       0.000000
max       0.900000
Name: confidence, dtype: float64

First 5 rows:
    sample_id  label    verdict  confidence  validity_score
0        672      1  UNCERTAIN         0.0             0.0
1       2294      1  UNCERTAIN         0.0             0.0
2        841      1  UNCERTAIN         0.0             0.0
3        758      1  UNCERTAIN         0.0             0.0
4        148      1   VERIFIED         0.9             0.9


In [47]:
import os, json
import pandas as pd
import numpy as np

# ---------- 1) Load full 2433 dataset ----------
df = pd.read_csv("megavul_rich_2433.csv")

# ensure id exists
if "sample_id" not in df.columns:
    df["sample_id"] = np.arange(len(df))

# build diff_all (strong evidence)
df["diff_all"] = df.get("diff_func","").astype(str) + " " + df.get("diff_line_info","").astype(str)

# ensure label int
df["is_vul"] = df["is_vul"].apply(_to_int_label)

# ---------- 2) Schema mapping ----------
schema = DatasetSchema(
    id_col="sample_id",
    code_col="func",              # fallback only (focus_code handled internally if you use that version)
    label_col="is_vul",
    commit_msg_col="commit_msg",
    diff_col="diff_all",
    before_code_col="func_before",
    after_code_col="func",
    cve_id_col="cve_id",
    cwe_col="cwe_ids",
    repo_col="repo_name",
    url_col="git_url",
    file_path_col="file_path",
    func_name_col="func_name",
    cve_desc_col=None
)

# ---------- 3) Output paths ----------
OUT_FULL   = "megavul_2433_scored_llm_rag_FULL.csv"
OUT_PART   = "megavul_2433_scored_llm_rag_FULL_partial.csv"

save_every = 50  # checkpoint frequency

# ---------- 4) Resume support ----------
done_ids = set()
if os.path.exists(OUT_PART):
    try:
        part = pd.read_csv(OUT_PART)
        if schema.id_col in part.columns:
            done_ids = set(part[schema.id_col].astype(int).tolist())
            print(f"[RESUME] Found partial with {len(done_ids)} done samples.")
    except Exception as e:
        print("[RESUME] Partial file exists but could not be read:", e)

# filter remaining
df_run = df[~df[schema.id_col].astype(int).isin(done_ids)].copy().reset_index(drop=True)
print("To score now:", len(df_run), "out of", len(df))

# ---------- 5) Initialize retriever + LLM ----------
retriever = RowLocalRetriever()   # uses E5 + (optional) reranker as in your class
llm = VerdictLLM()               # your chat-template + completion-only decode version

# If we resumed, load previous rows_out
rows_out = []
if os.path.exists(OUT_PART) and len(done_ids) > 0:
    # keep the already done rows in memory for final merge
    rows_out = part.to_dict("records")

# ---------- 6) Full loop ----------
for i, row in df_run.iterrows():
    retrieved = retriever.retrieve(row, schema, k=8)
    prompt = build_prompt(row, schema, retrieved)

    js = llm.generate(prompt, retrieved, max_new_tokens=220, retries=4, max_length=4096)

    verdict = str(js.get("verdict","UNCERTAIN")).upper().strip()
    conf = float(js.get("confidence", 0.0) or 0.0)
    ke = js.get("key_evidence", [])
    rat = js.get("rationale", "")

    validity = compute_validity_score(verdict, conf, retrieved, ke)

    rows_out.append({
        schema.id_col: int(row[schema.id_col]),
        "verdict": verdict,
        "confidence": conf,
        "validity_score": validity,
        "rationale": rat,
        "key_evidence": json.dumps(ke, ensure_ascii=False),
        "retrieved_types": ",".join([r["chunk_type"] for r in retrieved]) if retrieved else ""
    })

    # checkpoint
    if (len(rows_out) % save_every) == 0:
        pd.DataFrame(rows_out).to_csv(OUT_PART, index=False)
        print(f"[checkpoint] saved {len(rows_out)} rows -> {OUT_PART}")

# final save partial
pd.DataFrame(rows_out).to_csv(OUT_PART, index=False)
print("[done] final checkpoint saved ->", OUT_PART)

# ---------- 7) Merge scores back to full dataset ----------
score_df = pd.DataFrame(rows_out)

final = df.merge(score_df, on=schema.id_col, how="left")
final.to_csv(OUT_FULL, index=False)

print("[saved] full scored file ->", OUT_FULL)
print(final["verdict"].value_counts(dropna=False))
print(final["validity_score"].describe())


To score now: 2433 out of 2433
[checkpoint] saved 50 rows -> megavul_2433_scored_llm_rag_FULL_partial.csv
[checkpoint] saved 100 rows -> megavul_2433_scored_llm_rag_FULL_partial.csv
[checkpoint] saved 150 rows -> megavul_2433_scored_llm_rag_FULL_partial.csv
[checkpoint] saved 200 rows -> megavul_2433_scored_llm_rag_FULL_partial.csv
[checkpoint] saved 250 rows -> megavul_2433_scored_llm_rag_FULL_partial.csv
[checkpoint] saved 300 rows -> megavul_2433_scored_llm_rag_FULL_partial.csv
[checkpoint] saved 350 rows -> megavul_2433_scored_llm_rag_FULL_partial.csv
[checkpoint] saved 400 rows -> megavul_2433_scored_llm_rag_FULL_partial.csv
[checkpoint] saved 450 rows -> megavul_2433_scored_llm_rag_FULL_partial.csv
[checkpoint] saved 500 rows -> megavul_2433_scored_llm_rag_FULL_partial.csv
[checkpoint] saved 550 rows -> megavul_2433_scored_llm_rag_FULL_partial.csv
[checkpoint] saved 600 rows -> megavul_2433_scored_llm_rag_FULL_partial.csv
[checkpoint] saved 650 rows -> megavul_2433_scored_llm_rag

In [48]:
import pandas as pd

df = pd.read_csv("megavul_2433_scored_llm_rag_FULL_partial.csv")  # or your scored file

print(df["verdict"].value_counts(dropna=False))
print("\nPercent:")
print((df["verdict"].value_counts(normalize=True)*100).round(2))


verdict
UNCERTAIN       2394
VERIFIED          21
CONTRADICTED      18
Name: count, dtype: int64

Percent:
verdict
UNCERTAIN       98.40
VERIFIED         0.86
CONTRADICTED     0.74
Name: proportion, dtype: float64


3rd design with llamma

In [52]:
# ==========================================================
# Llama-3.2-3B-Instruct + Row-local RAG Validity Scoring
# FULL FINAL (debug-30 + full-run + checkpoints + mitigation boost)
# ==========================================================
import os, re, json
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional, Any

from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch


# ----------------------------
# 1) Schema (dataset-agnostic mapping)
# ----------------------------
@dataclass
class DatasetSchema:
    id_col: str = "sample_id"
    code_col: str = "code"      # fallback
    label_col: str = "label"    # 0/1 or bool

    commit_msg_col: Optional[str] = None
    diff_col: Optional[str] = None
    before_code_col: Optional[str] = None
    after_code_col: Optional[str] = None
    cve_id_col: Optional[str] = None
    cwe_col: Optional[str] = None
    repo_col: Optional[str] = None
    url_col: Optional[str] = None
    file_path_col: Optional[str] = None
    func_name_col: Optional[str] = None
    cve_desc_col: Optional[str] = None  # optional advisory text


def _clean(x) -> str:
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return ""
    s = str(x)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def _to_int_label(y) -> int:
    if isinstance(y, bool):
        return int(y)
    s = str(y).strip().lower()
    if s in ["1","true","t","yes","y"]: return 1
    if s in ["0","false","f","no","n"]: return 0
    try: return int(float(s))
    except: return 0

def _norm_ws(s: str) -> str:
    return re.sub(r"\s+", "", _clean(s))


# ----------------------------
# 2) Label-aware focus code (fixes ambiguity)
# ----------------------------
def get_focus_code(row: pd.Series, schema: DatasetSchema) -> str:
    """
    Patch-style datasets:
      label=1 -> vulnerable-before => use before_code_col if present
      label=0 -> fixed-after       => use after_code_col if present
    Fallback: schema.code_col
    """
    y = _to_int_label(row.get(schema.label_col, 0))
    if y == 1 and schema.before_code_col:
        v = _clean(row.get(schema.before_code_col, ""))
        if v:
            return v
    if y == 0 and schema.after_code_col:
        v = _clean(row.get(schema.after_code_col, ""))
        if v:
            return v
    return _clean(row.get(schema.code_col, ""))


# ----------------------------
# 3) Row-local chunking
# ----------------------------
def chunk_text(text: str, max_chars: int = 1200, overlap: int = 150) -> List[str]:
    text = _clean(text)
    if not text:
        return []
    if len(text) <= max_chars:
        return [text]
    out = []
    start = 0
    while start < len(text):
        end = min(len(text), start + max_chars)
        out.append(text[start:end].strip())
        if end == len(text):
            break
        start = max(0, end - overlap)
    return [c for c in out if c]

def build_chunks_for_row(row: pd.Series, schema: DatasetSchema,
                         max_chars: int = 1200, overlap: int = 150) -> List[Tuple[str,str]]:
    chunks: List[Tuple[str,str]] = []

    def add(tag: str, val: Any):
        for c in chunk_text(val, max_chars=max_chars, overlap=overlap):
            chunks.append((tag, c))

    # focus code (label-aware)
    add("focus_code", get_focus_code(row, schema))

    # evidence fields (if available)
    if schema.commit_msg_col:  add("commit_msg", row.get(schema.commit_msg_col, ""))
    if schema.diff_col:        add("diff", row.get(schema.diff_col, ""))
    if schema.before_code_col: add("before_code", row.get(schema.before_code_col, ""))
    if schema.after_code_col:  add("after_code", row.get(schema.after_code_col, ""))
    if schema.cve_desc_col:    add("cve_desc", row.get(schema.cve_desc_col, ""))

    meta_bits = []
    for col, tag in [
        (schema.cve_id_col, "cve_id"),
        (schema.cwe_col, "cwe"),
        (schema.repo_col, "repo"),
        (schema.file_path_col, "file"),
        (schema.func_name_col, "func"),
        (schema.url_col, "url"),
    ]:
        if col:
            v = _clean(row.get(col, ""))
            if v:
                meta_bits.append(f"{tag}: {v}")
    if meta_bits:
        chunks.append(("meta", " | ".join(meta_bits)))

    return [(t,x) for t,x in chunks if x]


def make_query(row: pd.Series, schema: DatasetSchema) -> str:
    y = _to_int_label(row.get(schema.label_col, 0))
    claim = "vulnerable-before (label=1)" if y == 1 else "fixed-after (label=0)"
    cve = _clean(row.get(schema.cve_id_col, "")) if schema.cve_id_col else ""
    cwe = _clean(row.get(schema.cwe_col, "")) if schema.cwe_col else ""
    fn  = _clean(row.get(schema.func_name_col, "")) if schema.func_name_col else ""
    return f"{claim}. {cve} {cwe} {fn}".strip()


# ----------------------------
# 4) Type-aware row-local retrieval
# ----------------------------
class RowLocalRetriever:
    def __init__(self,
                 emb_model_name="intfloat/e5-small-v2",
                 rerank_model_name: Optional[str] = "cross-encoder/ms-marco-MiniLM-L-6-v2"):
        self.emb = SentenceTransformer(emb_model_name)
        self.reranker = CrossEncoder(rerank_model_name) if rerank_model_name else None

    def retrieve(self, row: pd.Series, schema: DatasetSchema,
                 k: int = 8, max_chars: int = 1200, overlap: int = 150,
                 force_types=("diff","before_code","after_code","commit_msg","cve_desc","focus_code")) -> List[Dict[str,Any]]:
        chunks = build_chunks_for_row(row, schema, max_chars=max_chars, overlap=overlap)
        if not chunks:
            return []

        qtxt = make_query(row, schema)
        q = self.emb.encode([f"query: {qtxt}"], normalize_embeddings=True)
        X = self.emb.encode([f"passage: {c[1]}" for c in chunks], normalize_embeddings=True)

        sims = (X @ q[0]).astype(float)
        order = np.argsort(-sims)

        # overfetch candidates
        cand = []
        for i in order[:min(len(order), max(k*3, 24))]:
            cand.append({"doc_id": int(i), "chunk_type": chunks[i][0], "text": chunks[i][1], "score": float(sims[i])})

        # rerank top 24 only
        if self.reranker is not None and len(cand) > 1:
            pairs = [(qtxt, r["text"]) for r in cand[:min(len(cand), 24)]]
            rr = self.reranker.predict(pairs)
            for r, s in zip(cand[:len(pairs)], rr):
                r["rerank_score"] = float(s)
            cand = sorted(cand, key=lambda r: r.get("rerank_score", r["score"]), reverse=True)

        def key(r): return r.get("rerank_score", r["score"])

        # best per type
        best = {}
        for r in cand:
            t = r["chunk_type"]
            if (t not in best) or (key(r) > key(best[t])):
                best[t] = r

        out = []
        used = set()
        for t in force_types:
            if t in best:
                out.append(best[t]); used.add((best[t]["chunk_type"], best[t]["doc_id"]))

        for r in cand:
            kk = (r["chunk_type"], r["doc_id"])
            if kk not in used:
                out.append(r); used.add(kk)
            if len(out) >= k:
                break

        return out[:k]


# ----------------------------
# 5) Mitigation booster (fixes "no explicit security intent" issue)
# ----------------------------
FIX_PAT = re.compile(
    r"(bounds|out[- ]of[- ]bounds|oob|length|limit|sanitize|validation|validate|"
    r"auth|permission|csrf|xss|injection|!=\s*NULL|==\s*NULL|"
    r"strncpy|snprintf|memcpy|memmove|"
    r"escape|encode|canonicalize|"
    r"check\s*\(|if\s*\()",
    re.I
)

REFAC_PAT = re.compile(r"(refactor|format|cleanup|rename|typo|style|whitespace|lint|reorder)", re.I)

def mitigation_hit(row: pd.Series, schema: DatasetSchema) -> int:
    txt = ""
    if schema.diff_col: txt += " " + _clean(row.get(schema.diff_col, ""))
    if schema.commit_msg_col: txt += " " + _clean(row.get(schema.commit_msg_col, ""))
    txt += " " + get_focus_code(row, schema)
    return 1 if FIX_PAT.search(txt) else 0

def refactor_only_hit(row: pd.Series, schema: DatasetSchema) -> int:
    msg = _clean(row.get(schema.commit_msg_col, "")) if schema.commit_msg_col else ""
    diff = _clean(row.get(schema.diff_col, "")) if schema.diff_col else ""
    # commit says refactor/cleanup AND no fix patterns in diff
    if REFAC_PAT.search(msg) and (not FIX_PAT.search(diff)):
        return 1
    return 0


# ----------------------------
# 6) Llama-3.2 verifier (chat template + completion-only decode + JSON validate+retry)
# ----------------------------
def validate_json(js: dict, retrieved: List[Dict[str,Any]]) -> bool:
    if not isinstance(js, dict):
        return False
    verdict = str(js.get("verdict","")).upper().strip()
    if verdict not in ["VERIFIED","UNCERTAIN","CONTRADICTED"]:
        return False
    try:
        conf = float(js.get("confidence", None))
    except:
        return False
    if conf < 0 or conf > 1:
        return False

    ke = js.get("key_evidence", None)
    if not isinstance(ke, list) or len(ke) == 0:
        return False

    doc_map = {int(r["doc_id"]): r["text"] for r in retrieved}
    ok = False
    for ev in ke[:3]:
        if isinstance(ev, dict) and "doc_id" in ev and "snippet" in ev:
            try:
                did = int(ev["doc_id"])
            except:
                continue
            sn = str(ev["snippet"]).strip()
            if did in doc_map and len(sn) >= 10:
                ok = True
                break
    if not ok:
        return False
    return True


class LlamaVerdictLLM:
    def __init__(self, model_name="microsoft/Phi-3.5-mini-instruct"):
        self.model_name = model_name
        self.tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.tok.truncation_side = "left"  # keep JSON schema at end

        dtype = torch.float16 if torch.cuda.is_available() else torch.float32
        self.model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=dtype)

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = self.model.to(self.device)
        self.model.eval()

        if self.tok.pad_token_id is None:
            self.tok.pad_token = self.tok.eos_token

    def _format_chat(self, user_text: str) -> str:
        if hasattr(self.tok, "apply_chat_template"):
            msgs = [
                {"role": "system", "content": "You must output ONLY valid JSON. No extra text."},
                {"role": "user", "content": user_text},
            ]
            return self.tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        return user_text

    def _extract_json(self, text: str) -> Optional[dict]:
        text = text.strip()
        m = re.search(r"\{[\s\S]*\}", text)
        if not m:
            return None
        cand = m.group(0).strip()
        cand = re.sub(r",\s*}", "}", cand)
        cand = re.sub(r",\s*]", "]", cand)
        try:
            return json.loads(cand)
        except:
            return None

    def generate(self, user_prompt: str, retrieved: List[Dict[str,Any]],
                 max_new_tokens=220, retries=4, max_length=4096) -> dict:
        p = user_prompt
        for _ in range(retries + 1):
            prompt = self._format_chat(p)
            inputs = self.tok(prompt, return_tensors="pt", truncation=True, max_length=max_length).to(self.device)

            with torch.no_grad():
                out = self.model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=False,
                    temperature=0.0,
                    top_p=1.0,
                    eos_token_id=self.tok.eos_token_id,
                    pad_token_id=self.tok.pad_token_id
                )

            # completion-only decode
            prompt_len = inputs["input_ids"].shape[1]
            gen_ids = out[0][prompt_len:]
            text = self.tok.decode(gen_ids, skip_special_tokens=True).strip()

            js = self._extract_json(text)
            if js is not None and validate_json(js, retrieved):
                return js

            p = p + "\n\nIMPORTANT: Return ONLY JSON. Start with { and end with }. No extra text."

        return {"verdict":"UNCERTAIN","confidence":0.0,"rationale":"Invalid JSON output.","key_evidence":[]}


# ----------------------------
# 7) Prompt (VERIFIED widened: mitigation-only allowed)
# ----------------------------
def build_prompt(row: pd.Series, schema: DatasetSchema, retrieved: List[Dict[str,Any]]) -> str:
    y = _to_int_label(row.get(schema.label_col, 0))
    q = make_query(row, schema)

    # keep evidence bounded (avoid truncating schema)
    ctx = []
    for r in retrieved[:6]:
        ctx.append(f"[doc_id={r['doc_id']} | {r['chunk_type']}]\n{r['text'][:900]}\n")
    ctx = "\n---\n".join(ctx)

    return f"""Label claim: {y} (1=vulnerable-before, 0=fixed-after)
Query: {q}

Evidence:
{ctx}

Decision rules:
- VERIFIED if Evidence shows a mitigation change (bounds/NULL/length checks, input validation, sanitization, auth/permission checks, safer API).
  (Security intent mention is NOT required.)
- CONTRADICTED if Evidence indicates refactor/format/perf-only and no mitigation, or clearly implies the opposite label.
- Otherwise UNCERTAIN.

Return ONLY JSON exactly:
{{
  "verdict": "VERIFIED"|"UNCERTAIN"|"CONTRADICTED",
  "confidence": 0 to 1,
  "rationale": "1 short sentence",
  "key_evidence": [{{"doc_id": 0, "snippet": "copy exact substring from Evidence"}}]
}}

Rules:
- If verdict is UNCERTAIN, set confidence <= 0.6 (not too strict).
- key_evidence snippet must be copied from Evidence (no paraphrase).
"""


# ----------------------------
# 8) validity score (continuous, usable for weighted training)
# ----------------------------
def compute_validity_score(verdict: str, conf: float, row: pd.Series, schema: DatasetSchema) -> float:
    conf = float(np.clip(float(conf), 0.0, 1.0))
    # heuristic boosters
    m = mitigation_hit(row, schema)
    r = refactor_only_hit(row, schema)

    # base score from confidence
    score = conf

    # boost if mitigation found
    if m == 1:
        score = max(score, 0.65)

    # penalize if refactor-only signals
    if r == 1:
        score = min(score, 0.25)

    # cap uncertain moderately (not too harsh)
    if verdict == "UNCERTAIN":
        score = min(score, 0.60)

    return float(np.clip(score, 0.0, 1.0))


def postprocess_verdict(verdict: str, conf: float, row: pd.Series, schema: DatasetSchema) -> Tuple[str, float]:
    """
    Fix B) No explicit security intent: use mitigation patterns to convert some UNCERTAIN to VERIFIED.
    """
    conf = float(np.clip(float(conf), 0.0, 1.0))
    m = mitigation_hit(row, schema)
    r = refactor_only_hit(row, schema)

    # If LLM unsure but mitigation is obvious => VERIFIED (moderate confidence)
    if verdict == "UNCERTAIN" and m == 1 and r == 0:
        return "VERIFIED", max(conf, 0.65)

    # If refactor-only strong => CONTRADICTED (moderate confidence)
    if verdict == "UNCERTAIN" and r == 1 and m == 0:
        return "CONTRADICTED", max(min(conf, 0.35), 0.30)

    return verdict, conf


# ----------------------------
# 9) DEBUG 30 first
# ----------------------------
def debug_30(df: pd.DataFrame, schema: DatasetSchema, seed=0, out_csv="debug_30_llm_rag_llama.csv"):
    if schema.id_col not in df.columns:
        df[schema.id_col] = np.arange(len(df))
    df[schema.label_col] = df[schema.label_col].apply(_to_int_label)

    pos = df[df[schema.label_col]==1]
    neg = df[df[schema.label_col]==0]
    n_pos = min(15, len(pos))
    n_neg = min(15, len(neg))

    if n_pos == 0 or n_neg == 0:
        df30 = df.sample(min(30, len(df)), random_state=seed).reset_index(drop=True)
    else:
        df30 = pd.concat([pos.sample(n_pos, random_state=seed),
                          neg.sample(n_neg, random_state=seed)], axis=0)\
               .sample(frac=1, random_state=seed).reset_index(drop=True)

    retriever = RowLocalRetriever()
    llm = LlamaVerdictLLM()

    rows = []
    for _, row in df30.iterrows():
        retrieved = retriever.retrieve(row, schema, k=8)
        prompt = build_prompt(row, schema, retrieved)
        js = llm.generate(prompt, retrieved, max_new_tokens=220, retries=4, max_length=4096)

        verdict = str(js.get("verdict","UNCERTAIN")).upper().strip()
        conf = float(js.get("confidence", 0.0) or 0.0)

        verdict, conf = postprocess_verdict(verdict, conf, row, schema)
        vscore = compute_validity_score(verdict, conf, row, schema)

        rows.append({
            schema.id_col: int(row[schema.id_col]),
            "label": int(row[schema.label_col]),
            "verdict": verdict,
            "confidence": conf,
            "validity_score": vscore,
            "rationale": js.get("rationale",""),
            "key_evidence": json.dumps(js.get("key_evidence", []), ensure_ascii=False),
            "retrieved_types": ",".join([r["chunk_type"] for r in retrieved]) if retrieved else ""
        })

    out = pd.DataFrame(rows)
    out.to_csv(out_csv, index=False)
    print("Saved:", out_csv)
    print(out["verdict"].value_counts())
    print(out["confidence"].describe())
    print(out["validity_score"].describe())
    return out


# ----------------------------
# 10) FULL scoring with resume + checkpoints
# ----------------------------
def score_full(df: pd.DataFrame, schema: DatasetSchema,
               out_full="megavul_2433_scored_llm_rag_LLAMA_FULL.csv",
               out_part="megavul_2433_scored_llm_rag_LLAMA_partial.csv",
               save_every=50):
    if schema.id_col not in df.columns:
        df[schema.id_col] = np.arange(len(df))
    df[schema.label_col] = df[schema.label_col].apply(_to_int_label)

    done_ids = set()
    rows_out = []

    if os.path.exists(out_part):
        try:
            part = pd.read_csv(out_part)
            done_ids = set(part[schema.id_col].astype(int).tolist())
            rows_out = part.to_dict("records")
            print(f"[RESUME] loaded {len(done_ids)} done rows from {out_part}")
        except Exception as e:
            print("[RESUME] could not read partial:", e)

    df_run = df[~df[schema.id_col].astype(int).isin(done_ids)].copy().reset_index(drop=True)
    print("To score:", len(df_run), "/", len(df))

    retriever = RowLocalRetriever()
    llm = LlamaVerdictLLM()

    for i, row in df_run.iterrows():
        retrieved = retriever.retrieve(row, schema, k=8)
        prompt = build_prompt(row, schema, retrieved)
        js = llm.generate(prompt, retrieved, max_new_tokens=220, retries=4, max_length=4096)

        verdict = str(js.get("verdict","UNCERTAIN")).upper().strip()
        conf = float(js.get("confidence", 0.0) or 0.0)

        verdict, conf = postprocess_verdict(verdict, conf, row, schema)
        vscore = compute_validity_score(verdict, conf, row, schema)

        rows_out.append({
            schema.id_col: int(row[schema.id_col]),
            "verdict": verdict,
            "confidence": conf,
            "validity_score": vscore,
            "rationale": js.get("rationale",""),
            "key_evidence": json.dumps(js.get("key_evidence", []), ensure_ascii=False),
            "retrieved_types": ",".join([r["chunk_type"] for r in retrieved]) if retrieved else ""
        })

        if (len(rows_out) % save_every) == 0:
            pd.DataFrame(rows_out).to_csv(out_part, index=False)
            print(f"[checkpoint] {len(rows_out)} rows saved -> {out_part}")

    pd.DataFrame(rows_out).to_csv(out_part, index=False)

    score_df = pd.DataFrame(rows_out)
    final = df.merge(score_df, on=schema.id_col, how="left")
    final.to_csv(out_full, index=False)
    print("[saved]", out_full)
    print(final["verdict"].value_counts(dropna=False))
    print(final["validity_score"].describe())
    return final


# ==========================================================
# MEGAVUL 2433: RUN
# ==========================================================
df = pd.read_csv("megavul_rich_2433.csv")

# ensure id
if "sample_id" not in df.columns:
    df["sample_id"] = np.arange(len(df))

# build diff_all
df["diff_all"] = df.get("diff_func","").astype(str) + " " + df.get("diff_line_info","").astype(str)

# label int
df["is_vul"] = df["is_vul"].apply(_to_int_label)

schema = DatasetSchema(
    id_col="sample_id",
    code_col="func",          # fallback only
    label_col="is_vul",
    commit_msg_col="commit_msg",
    diff_col="diff_all",
    before_code_col="func_before",
    after_code_col="func",
    cve_id_col="cve_id",
    cwe_col="cwe_ids",
    repo_col="repo_name",
    url_col="git_url",
    file_path_col="file_path",
    func_name_col="func_name",
    cve_desc_col=None
)

# 1) Debug 30
dbg = debug_30(df, schema, seed=0, out_csv="debug_30_llm_rag_llama.csv")

# 2) Full run (uncomment)
# full = score_full(df, schema)


Loading checkpoint shards: 100%|██████████| 2/2 [00:02<00:00,  1.09s/it]
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Saved: debug_30_llm_rag_llama.csv
verdict
VERIFIED    30
Name: count, dtype: int64
count    30.000000
mean      0.923333
std       0.126446
min       0.650000
25%       0.900000
50%       1.000000
75%       1.000000
max       1.000000
Name: confidence, dtype: float64
count    30.000000
mean      0.923333
std       0.126446
min       0.650000
25%       0.900000
50%       1.000000
75%       1.000000
max       1.000000
Name: validity_score, dtype: float64


In [53]:
import pandas as pd

df = pd.read_csv("debug_30_llm_rag_llama.csv")  # or your scored file

print(df["verdict"].value_counts(dropna=False))
print("\nPercent:")
print((df["verdict"].value_counts(normalize=True)*100).round(2))


verdict
VERIFIED    30
Name: count, dtype: int64

Percent:
verdict
VERIFIED    100.0
Name: proportion, dtype: float64


In [54]:
# ==========================================================
# FULL DATASET RUN (2433 rows): LLM + RAG Validity Scoring ✅
# - Works on megavul_rich_2433.csv
# - Saves partial checkpoints + final merged CSV
# - Resume supported (if partial exists)
# ==========================================================

import os, re, json
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional, Any

from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# =========================
# CONFIG
# =========================
MODEL_NAME   = "microsoft/Phi-3.5-mini-instruct"   # open + reliable
EMB_MODEL    = "intfloat/e5-small-v2"
RERANK_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"  # set None to disable
TOPK = 8
MAX_NEW_TOKENS = 220
MAX_PROMPT_LEN = 4096
RETRIES = 4
SAVE_EVERY = 50

OUT_FULL = "megavul_2433_scored_llm_rag_FULL.csv"
OUT_PART = "megavul_2433_scored_llm_rag_partial.csv"


# ----------------------------
# Schema
# ----------------------------
@dataclass
class DatasetSchema:
    id_col: str = "sample_id"
    code_col: str = "code"
    label_col: str = "label"
    commit_msg_col: Optional[str] = None
    diff_col: Optional[str] = None
    before_code_col: Optional[str] = None
    after_code_col: Optional[str] = None
    cve_id_col: Optional[str] = None
    cwe_col: Optional[str] = None
    repo_col: Optional[str] = None
    url_col: Optional[str] = None
    file_path_col: Optional[str] = None
    func_name_col: Optional[str] = None
    cve_desc_col: Optional[str] = None


def _clean(x) -> str:
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return ""
    s = str(x)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def _to_int_label(y) -> int:
    if isinstance(y, bool):
        return int(y)
    s = str(y).strip().lower()
    if s in ["1","true","t","yes","y"]: return 1
    if s in ["0","false","f","no","n"]: return 0
    try: return int(float(s))
    except: return 0


# ----------------------------
# Focus code (label-aware)
# ----------------------------
def get_focus_code(row: pd.Series, schema: DatasetSchema) -> str:
    y = _to_int_label(row.get(schema.label_col, 0))
    if y == 1 and schema.before_code_col:
        v = _clean(row.get(schema.before_code_col, ""))
        if v: return v
    if y == 0 and schema.after_code_col:
        v = _clean(row.get(schema.after_code_col, ""))
        if v: return v
    return _clean(row.get(schema.code_col, ""))


# ----------------------------
# Chunking + evidence pack
# ----------------------------
def chunk_text(text: str, max_chars: int = 1200, overlap: int = 150) -> List[str]:
    text = _clean(text)
    if not text:
        return []
    if len(text) <= max_chars:
        return [text]
    out = []
    start = 0
    while start < len(text):
        end = min(len(text), start + max_chars)
        out.append(text[start:end].strip())
        if end == len(text):
            break
        start = max(0, end - overlap)
    return [c for c in out if c]

def build_chunks_for_row(row: pd.Series, schema: DatasetSchema) -> List[Tuple[str,str]]:
    chunks: List[Tuple[str,str]] = []
    def add(tag, val):
        for c in chunk_text(val):
            chunks.append((tag, c))

    add("focus_code", get_focus_code(row, schema))
    if schema.commit_msg_col:  add("commit_msg", row.get(schema.commit_msg_col, ""))
    if schema.diff_col:        add("diff", row.get(schema.diff_col, ""))
    if schema.before_code_col: add("before_code", row.get(schema.before_code_col, ""))
    if schema.after_code_col:  add("after_code", row.get(schema.after_code_col, ""))
    if schema.cve_desc_col:    add("cve_desc", row.get(schema.cve_desc_col, ""))

    meta = []
    for col, tag in [
        (schema.cve_id_col, "cve_id"),
        (schema.cwe_col, "cwe"),
        (schema.repo_col, "repo"),
        (schema.file_path_col, "file"),
        (schema.func_name_col, "func"),
        (schema.url_col, "url"),
    ]:
        if col:
            v = _clean(row.get(col, ""))
            if v:
                meta.append(f"{tag}: {v}")
    if meta:
        chunks.append(("meta", " | ".join(meta)))

    return [(t,x) for t,x in chunks if x]


def make_query(row: pd.Series, schema: DatasetSchema) -> str:
    y = _to_int_label(row.get(schema.label_col, 0))
    claim = "vulnerable-before (label=1)" if y == 1 else "fixed-after (label=0)"
    cve = _clean(row.get(schema.cve_id_col, "")) if schema.cve_id_col else ""
    cwe = _clean(row.get(schema.cwe_col, "")) if schema.cwe_col else ""
    fn  = _clean(row.get(schema.func_name_col, "")) if schema.func_name_col else ""
    return f"{claim}. {cve} {cwe} {fn}".strip()


# ----------------------------
# Retriever (type-aware)
# ----------------------------
class RowLocalRetriever:
    def __init__(self, emb_model_name=EMB_MODEL, rerank_model_name=RERANK_MODEL):
        self.emb = SentenceTransformer(emb_model_name)
        self.reranker = CrossEncoder(rerank_model_name) if rerank_model_name else None

    def retrieve(self, row: pd.Series, schema: DatasetSchema, k=TOPK,
                 force_types=("diff","before_code","after_code","commit_msg","cve_desc","focus_code")):
        chunks = build_chunks_for_row(row, schema)
        if not chunks:
            return []
        qtxt = make_query(row, schema)

        q = self.emb.encode([f"query: {qtxt}"], normalize_embeddings=True)
        X = self.emb.encode([f"passage: {c[1]}" for c in chunks], normalize_embeddings=True)

        sims = (X @ q[0]).astype(float)
        order = np.argsort(-sims)

        cand = []
        for i in order[:min(len(order), max(k*3, 24))]:
            cand.append({"doc_id": int(i), "chunk_type": chunks[i][0], "text": chunks[i][1], "score": float(sims[i])})

        if self.reranker is not None and len(cand) > 1:
            pairs = [(qtxt, r["text"]) for r in cand[:min(len(cand), 24)]]
            rr = self.reranker.predict(pairs)
            for r, s in zip(cand[:len(pairs)], rr):
                r["rerank_score"] = float(s)
            cand = sorted(cand, key=lambda r: r.get("rerank_score", r["score"]), reverse=True)

        def key(r): return r.get("rerank_score", r["score"])
        best = {}
        for r in cand:
            t = r["chunk_type"]
            if (t not in best) or (key(r) > key(best[t])):
                best[t] = r

        out = []
        used = set()
        for t in force_types:
            if t in best:
                out.append(best[t]); used.add((best[t]["chunk_type"], best[t]["doc_id"]))
        for r in cand:
            kk = (r["chunk_type"], r["doc_id"])
            if kk not in used:
                out.append(r); used.add(kk)
            if len(out) >= k:
                break
        return out[:k]


# ----------------------------
# LLM verifier (JSON stable)
# ----------------------------
def validate_json(js: dict, retrieved: List[Dict[str,Any]]) -> bool:
    if not isinstance(js, dict):
        return False
    verdict = str(js.get("verdict","")).upper().strip()
    if verdict not in ["VERIFIED","UNCERTAIN","CONTRADICTED"]:
        return False
    try:
        conf = float(js.get("confidence", None))
    except:
        return False
    if conf < 0 or conf > 1:
        return False
    ke = js.get("key_evidence", None)
    if not isinstance(ke, list) or len(ke) == 0:
        return False
    doc_ids = {int(r["doc_id"]) for r in retrieved}
    ok = False
    for ev in ke[:3]:
        if isinstance(ev, dict) and "doc_id" in ev and "snippet" in ev:
            try:
                did = int(ev["doc_id"])
            except:
                continue
            sn = str(ev["snippet"]).strip()
            if did in doc_ids and len(sn) >= 10:
                ok = True
                break
    return ok


class VerdictLLM:
    def __init__(self, model_name=MODEL_NAME):
        self.tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.tok.truncation_side = "left"  # keep schema at end

        dtype = torch.float16 if torch.cuda.is_available() else torch.float32
        self.model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=dtype)

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = self.model.to(self.device)
        self.model.eval()

        if self.tok.pad_token_id is None:
            self.tok.pad_token = self.tok.eos_token

    def _format_chat(self, user_text: str) -> str:
        if hasattr(self.tok, "apply_chat_template"):
            msgs = [
                {"role": "system", "content": "Return ONLY valid JSON. No extra text."},
                {"role": "user", "content": user_text},
            ]
            return self.tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        return user_text

    def _extract_json(self, text: str) -> Optional[dict]:
        text = text.strip()
        m = re.search(r"\{[\s\S]*\}", text)
        if not m:
            return None
        cand = m.group(0).strip()
        cand = re.sub(r",\s*}", "}", cand)
        cand = re.sub(r",\s*]", "]", cand)
        try:
            return json.loads(cand)
        except:
            return None

    def generate(self, user_prompt: str, retrieved: List[Dict[str,Any]]):
        p = user_prompt
        for _ in range(RETRIES + 1):
            prompt = self._format_chat(p)
            inputs = self.tok(prompt, return_tensors="pt", truncation=True, max_length=MAX_PROMPT_LEN).to(self.device)

            with torch.no_grad():
                out = self.model.generate(
                    **inputs,
                    max_new_tokens=MAX_NEW_TOKENS,
                    do_sample=False,
                    temperature=0.0,
                    top_p=1.0,
                    eos_token_id=self.tok.eos_token_id,
                    pad_token_id=self.tok.pad_token_id
                )

            # completion-only decode
            prompt_len = inputs["input_ids"].shape[1]
            gen_ids = out[0][prompt_len:]
            text = self.tok.decode(gen_ids, skip_special_tokens=True).strip()

            js = self._extract_json(text)
            if js is not None and validate_json(js, retrieved):
                return js

            p = p + "\n\nIMPORTANT: Output ONLY JSON. Start with { and end with }. No extra text."
        return {"verdict":"UNCERTAIN","confidence":0.0,"rationale":"Invalid JSON output.","key_evidence":[]}


def build_prompt(row: pd.Series, schema: DatasetSchema, retrieved: List[Dict[str,Any]]) -> str:
    y = _to_int_label(row.get(schema.label_col, 0))
    q = make_query(row, schema)

    ctx = []
    for r in retrieved[:6]:
        ctx.append(f"[doc_id={r['doc_id']} | {r['chunk_type']}]\n{r['text'][:900]}\n")
    ctx = "\n---\n".join(ctx)

    return f"""Label claim: {y} (1=vulnerable-before, 0=fixed-after)
Query: {q}

Evidence:
{ctx}

Decision rules:
- VERIFIED if Evidence shows mitigation change (NULL/bounds checks, safer API replacement, sanitization, auth/permission checks, CVE/CWE/security cues).
- CONTRADICTED if Evidence indicates refactor/format/performance-only and no mitigation.
- Otherwise UNCERTAIN.

Constraint: If UNCERTAIN, confidence MUST be <= 0.6.

Return ONLY JSON exactly:
{{
  "verdict": "VERIFIED"|"UNCERTAIN"|"CONTRADICTED",
  "confidence": 0 to 1,
  "rationale": "1 short sentence",
  "key_evidence": [{{"doc_id": 0, "snippet": "copy from Evidence"}}]
}}
"""


# ----------------------------
# Full-run function
# ----------------------------
def score_full(df: pd.DataFrame, schema: DatasetSchema):
    # resume support
    done_ids = set()
    rows_out = []

    if os.path.exists(OUT_PART):
        part = pd.read_csv(OUT_PART)
        if schema.id_col in part.columns:
            done_ids = set(part[schema.id_col].astype(int).tolist())
            rows_out = part.to_dict("records")
            print(f"[RESUME] Loaded {len(done_ids)} done rows.")

    df_run = df[~df[schema.id_col].astype(int).isin(done_ids)].copy().reset_index(drop=True)
    print("To score:", len(df_run), "/", len(df))

    retriever = RowLocalRetriever()
    llm = VerdictLLM()

    for i, row in df_run.iterrows():
        retrieved = retriever.retrieve(row, schema, k=TOPK)
        prompt = build_prompt(row, schema, retrieved)
        js = llm.generate(prompt, retrieved)

        verdict = str(js.get("verdict","UNCERTAIN")).upper().strip()
        conf = float(js.get("confidence", 0.0) or 0.0)
        rat = js.get("rationale", "")
        ke  = js.get("key_evidence", [])

        rows_out.append({
            schema.id_col: int(row[schema.id_col]),
            "verdict": verdict,
            "confidence": conf,
            "validity_score": conf,  # you can later add heuristic boosts if needed
            "rationale": rat,
            "key_evidence": json.dumps(ke, ensure_ascii=False),
            "retrieved_types": ",".join([r["chunk_type"] for r in retrieved]) if retrieved else ""
        })

        if (len(rows_out) % SAVE_EVERY) == 0:
            pd.DataFrame(rows_out).to_csv(OUT_PART, index=False)
            print(f"[checkpoint] saved {len(rows_out)} rows -> {OUT_PART}")

    # save final checkpoint
    pd.DataFrame(rows_out).to_csv(OUT_PART, index=False)

    score_df = pd.DataFrame(rows_out)
    final = df.merge(score_df, on=schema.id_col, how="left")
    final.to_csv(OUT_FULL, index=False)

    print("[saved]", OUT_FULL)
    print(final["verdict"].value_counts(dropna=False))
    print(final["confidence"].describe())
    return final


# ==========================================================
# RUN FULL 2433 ON MEGAVUL
# ==========================================================
df = pd.read_csv("megavul_rich_2433.csv")

# ensure id
if "sample_id" not in df.columns:
    df["sample_id"] = np.arange(len(df))

# build diff_all
df["diff_all"] = df.get("diff_func","").astype(str) + " " + df.get("diff_line_info","").astype(str)

# ensure label int
df["is_vul"] = df["is_vul"].apply(_to_int_label)

schema = DatasetSchema(
    id_col="sample_id",
    code_col="func",
    label_col="is_vul",
    commit_msg_col="commit_msg",
    diff_col="diff_all",
    before_code_col="func_before",
    after_code_col="func",
    cve_id_col="cve_id",
    cwe_col="cwe_ids",
    repo_col="repo_name",
    url_col="git_url",
    file_path_col="file_path",
    func_name_col="func_name",
    cve_desc_col=None
)

# FULL RUN
final = score_full(df, schema)


To score: 2433 / 2433


Loading checkpoint shards: 100%|██████████| 2/2 [00:04<00:00,  2.33s/it]


[checkpoint] saved 50 rows -> megavul_2433_scored_llm_rag_partial.csv
[checkpoint] saved 100 rows -> megavul_2433_scored_llm_rag_partial.csv
[checkpoint] saved 150 rows -> megavul_2433_scored_llm_rag_partial.csv
[checkpoint] saved 200 rows -> megavul_2433_scored_llm_rag_partial.csv
[checkpoint] saved 250 rows -> megavul_2433_scored_llm_rag_partial.csv
[checkpoint] saved 300 rows -> megavul_2433_scored_llm_rag_partial.csv
[checkpoint] saved 350 rows -> megavul_2433_scored_llm_rag_partial.csv
[checkpoint] saved 400 rows -> megavul_2433_scored_llm_rag_partial.csv
[checkpoint] saved 450 rows -> megavul_2433_scored_llm_rag_partial.csv
[checkpoint] saved 500 rows -> megavul_2433_scored_llm_rag_partial.csv
[checkpoint] saved 550 rows -> megavul_2433_scored_llm_rag_partial.csv
[checkpoint] saved 600 rows -> megavul_2433_scored_llm_rag_partial.csv
[checkpoint] saved 650 rows -> megavul_2433_scored_llm_rag_partial.csv
[checkpoint] saved 700 rows -> megavul_2433_scored_llm_rag_partial.csv
[checkp

neg verifyer

In [5]:
df=pd.read_csv("megavul_2433_scored_llm_rag_FULL.csv")

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2433 entries, 0 to 2432
Data columns (total 22 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   cve_id              2433 non-null   object 
 1   cwe_ids             2433 non-null   object 
 2   repo_name           2433 non-null   object 
 3   commit_msg          2433 non-null   object 
 4   commit_hash         2433 non-null   object 
 5   parent_commit_hash  2433 non-null   object 
 6   git_url             2433 non-null   object 
 7   file_path           2433 non-null   object 
 8   func_name           2433 non-null   object 
 9   func_before         2433 non-null   object 
 10  func                2433 non-null   object 
 11  diff_func           2433 non-null   object 
 12  diff_line_info      2433 non-null   object 
 13  is_vul              2433 non-null   int64  
 14  sample_id           2433 non-null   int64  
 15  diff_all            2433 non-null   object 
 16  verdic

In [7]:
import pandas as pd
import numpy as np

# যদি df_scored নামে আপনার dataframe থাকে, তাহলে এই লাইন লাগবে না
# df_scored = pd.read_csv("megavul_2433_scored_llm_rag_FULL.csv")

df_scored = df  # আপনি যে df দেখাচ্ছেন সেটাকেই ধরলাম

# -------------------------
# 1) retrieved_types: unique chunk types + frequency
# -------------------------
types_series = (
    df_scored["retrieved_types"]
    .fillna("")
    .astype(str)
    .str.split(",")
    .explode()
    .str.strip()
)

types_series = types_series[types_series != ""]
print("\nUnique retrieved chunk types:", types_series.nunique())
print("\nTop retrieved types:\n", types_series.value_counts().head(20))

# একসাথে কোন কোন combination বেশি আসে (optional)
print("\nTop retrieved_types combinations:\n", df_scored["retrieved_types"].value_counts().head(15))

# -------------------------
# 2) confidence + validity_score summary
# -------------------------
print("\nConfidence summary:\n", df_scored["confidence"].describe())
print("\nValidity score summary:\n", df_scored["validity_score"].describe())

# quantiles
qs = [0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95]
print("\nConfidence quantiles:\n", df_scored["confidence"].quantile(qs))
print("\nValidity quantiles:\n", df_scored["validity_score"].quantile(qs))

# bucket counts (histogram without plotting)
bins = [0, 0.3, 0.6, 0.8, 0.9, 0.95, 1.0]
labels = ["0-0.3","0.3-0.6","0.6-0.8","0.8-0.9","0.9-0.95","0.95-1.0"]
conf_bins = pd.cut(df_scored["confidence"], bins=bins, labels=labels, include_lowest=True)
val_bins  = pd.cut(df_scored["validity_score"], bins=bins, labels=labels, include_lowest=True)

print("\nConfidence buckets:\n", conf_bins.value_counts().sort_index())
print("\nValidity buckets:\n",  val_bins.value_counts().sort_index())

# verdict অনুযায়ী average confidence/validity
print("\nMean confidence by verdict:\n", df_scored.groupby("verdict")["confidence"].mean())
print("\nMean validity_score by verdict:\n", df_scored.groupby("verdict")["validity_score"].mean())



Unique retrieved chunk types: 6

Top retrieved types:
 retrieved_types
before_code    2875
after_code     2852
focus_code     2847
diff           2834
commit_msg     2438
meta           2433
Name: count, dtype: int64

Top retrieved_types combinations:
 retrieved_types
diff,before_code,after_code,commit_msg,focus_code,meta                           1479
diff,before_code,after_code,commit_msg,focus_code,meta,diff                       167
diff,before_code,after_code,commit_msg,focus_code,meta,before_code,focus_code     147
diff,before_code,after_code,commit_msg,focus_code,meta,after_code,before_code     134
diff,before_code,after_code,commit_msg,focus_code,meta,focus_code,before_code     116
diff,before_code,after_code,commit_msg,focus_code,meta,after_code,focus_code      108
diff,before_code,after_code,commit_msg,focus_code,meta,diff,diff                   67
diff,before_code,after_code,commit_msg,focus_code,meta,diff,after_code             40
diff,before_code,after_code,commit_msg,foc

In [8]:
import pandas as pd

in_path  = "megavul_2433_scored_llm_rag_FULL.csv"
out_path = "megavul_2433_scored_llm_rag_FULL_verified_only.csv"

df = pd.read_csv(in_path)

# keep only VERIFIED
df_verified = df[df["verdict"].astype(str).str.upper().str.strip() == "VERIFIED"].copy()

print("Original rows:", len(df))
print("Verified rows:", len(df_verified))
print(df_verified["verdict"].value_counts(dropna=False))

# save
df_verified.to_csv(out_path, index=False)
print("Saved:", out_path)


Original rows: 2433
Verified rows: 2041
verdict
VERIFIED    2041
Name: count, dtype: int64
Saved: megavul_2433_scored_llm_rag_FULL_verified_only.csv


In [9]:
df = pd.read_json("megavul (2).json")

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41949 entries, 0 to 41948
Data columns (total 32 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   cve_id                           41949 non-null  object 
 1   cwe_ids                          41949 non-null  object 
 2   cvss_vector                      41949 non-null  object 
 3   cvss_base_score                  41949 non-null  float64
 4   cvss_base_severity               41949 non-null  object 
 5   cvss_is_v3                       41949 non-null  bool   
 6   publish_date                     41949 non-null  object 
 7   repo_name                        41949 non-null  object 
 8   commit_msg                       41949 non-null  object 
 9   commit_hash                      41949 non-null  object 
 10  parent_commit_hash               41949 non-null  object 
 11  commit_date                      41949 non-null  int64  
 12  git_url           

In [12]:
import pandas as pd
import numpy as np

# -------- load --------
df = pd.read_json("megavul (2).json")

# -------- keep only these columns --------
keep_cols = [
    "cve_id",
    "cwe_ids",
    "repo_name",
    "commit_msg",
    "commit_hash",
    "parent_commit_hash",
    "git_url",
    "file_path",
    "func_name",
    "func_before",
    "func",
    "diff_func",
    "diff_line_info",
    "is_vul",
    "sample_id",
    "diff_all",
]

# keep only columns that exist (safe)
keep_cols = [c for c in keep_cols if c in df.columns]
df_small = df[keep_cols].copy()

# -------- ensure sample_id exists --------
if "sample_id" not in df_small.columns:
    df_small["sample_id"] = np.arange(len(df_small), dtype=int)

# -------- is_vul -> int (False=0, True=1) --------
df_small["is_vul"] = df_small["is_vul"].astype(int)

# -------- sample 2000 negatives --------
neg = df_small[df_small["is_vul"] == 0].copy()
print("Total negatives available:", len(neg))

n = 2000
if len(neg) < n:
    raise ValueError(f"Not enough negative samples. Found {len(neg)} but need {n}.")

neg_2000 = neg.sample(n=n, random_state=42).reset_index(drop=True)

# -------- save --------
neg_2000.to_csv("megavul_neg_2000_sample.csv", index=False)
neg_2000.to_json("megavul_neg_2000_sample.json", orient="records", force_ascii=False)

print("Saved:")
print(" - megavul_neg_2000_sample.csv")
print(" - megavul_neg_2000_sample.json")
print("Rows:", len(neg_2000), "| is_vul counts:\n", neg_2000["is_vul"].value_counts())


Total negatives available: 39516
Saved:
 - megavul_neg_2000_sample.csv
 - megavul_neg_2000_sample.json
Rows: 2000 | is_vul counts:
 is_vul
0    2000
Name: count, dtype: int64


In [11]:
import pandas as pd

in_path  = "megavul_2433_scored_llm_rag_FULL_verified_only.csv"
out_path = "megavul_2433_verified_only_NO_SCORES.csv"

df = pd.read_csv(in_path)

drop_cols = ["verdict","confidence","validity_score","rationale","key_evidence","retrieved_types"]
drop_cols = [c for c in drop_cols if c in df.columns]  # safe if missing

df2 = df.drop(columns=drop_cols)

print("Before:", df.shape)
print("After :", df2.shape)

df2.to_csv(out_path, index=False)
print("Saved:", out_path)


Before: (2041, 22)
After : (2041, 16)
Saved: megavul_2433_verified_only_NO_SCORES.csv


In [13]:
import pandas as pd
import numpy as np

pos_path = "megavul_2433_verified_only_NO_SCORES.csv"
neg_path = "megavul_neg_2000_sample.csv"
out_path = "megavul_pos2433_verified_plus_neg2000.csv"

df_pos = pd.read_csv(pos_path)
df_neg = pd.read_csv(neg_path)

# 1) Make sure is_vul is int 0/1
if "is_vul" in df_pos.columns:
    df_pos["is_vul"] = df_pos["is_vul"].astype(int)
if "is_vul" in df_neg.columns:
    df_neg["is_vul"] = df_neg["is_vul"].astype(int)

# 2) Align NEG to POS columns (POS defines schema + order)
pos_cols = list(df_pos.columns)

# add missing columns in neg as NaN
for c in pos_cols:
    if c not in df_neg.columns:
        df_neg[c] = np.nan

# drop extra columns that are not in pos
df_neg = df_neg[pos_cols]

# 3) Append (stack)
df_full = pd.concat([df_pos, df_neg], ignore_index=True)

print("POS rows:", len(df_pos))
print("NEG rows:", len(df_neg))
print("FULL rows:", len(df_full))
print("is_vul counts:\n", df_full["is_vul"].value_counts(dropna=False))

# 4) Save
df_full.to_csv(out_path, index=False)
print("Saved:", out_path)


POS rows: 2041
NEG rows: 2000
FULL rows: 4041
is_vul counts:
 is_vul
1    2041
0    2000
Name: count, dtype: int64
Saved: megavul_pos2433_verified_plus_neg2000.csv


version 4

In [1]:
# ==========================================================
# FULL DATASET RUN (2433 rows): LLM + RAG Validity Scoring ✅
# FIXED for: AttributeError 'DynamicCache' object has no attribute 'seen_tokens'
# - DO NOT use trust_remote_code (prevents loading outdated modeling_phi3.py)
# - Use use_cache=False in generate as extra safety
# - Fresh run by default (no old partial contamination)
# - LLM predicts predicted_label + pred_confidence (label NOT shown)
# - verdict computed programmatically (VERIFIED/UNCERTAIN/CONTRADICTED)
# ==========================================================

import os, re, json, time
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional, Any

from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch


# =========================
# CONFIG
# =========================
MODEL_NAME   = "microsoft/Phi-3.5-mini-instruct"
EMB_MODEL    = "intfloat/e5-small-v2"
RERANK_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"  # set None to disable

TOPK = 8
MAX_NEW_TOKENS = 220
MAX_PROMPT_LEN = 4096
RETRIES = 4
SAVE_EVERY = 50

FORCE_FRESH_RUN = True
RUN_TAG = time.strftime("%Y%m%d_%H%M%S")
OUT_FULL = f"megavul_2433_scored_llm_rag_FULL_{RUN_TAG}.csv"
OUT_PART = f"megavul_2433_scored_llm_rag_partial_{RUN_TAG}.csv"


# ----------------------------
# Schema
# ----------------------------
@dataclass
class DatasetSchema:
    id_col: str = "sample_id"
    code_col: str = "code"
    label_col: str = "label"
    commit_msg_col: Optional[str] = None
    diff_col: Optional[str] = None
    before_code_col: Optional[str] = None
    after_code_col: Optional[str] = None
    cve_id_col: Optional[str] = None
    cwe_col: Optional[str] = None
    repo_col: Optional[str] = None
    url_col: Optional[str] = None
    file_path_col: Optional[str] = None
    func_name_col: Optional[str] = None
    cve_desc_col: Optional[str] = None


# ----------------------------
# Helpers
# ----------------------------
def _as_str(x) -> str:
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return ""
    return str(x)

def _clean_nl(x) -> str:
    s = _as_str(x)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def _clean_code(x) -> str:
    s = _as_str(x)
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    s = "\n".join([ln.rstrip() for ln in s.split("\n")]).strip()
    return s

def _to_int_label(y) -> int:
    if isinstance(y, bool):
        return int(y)
    s = _as_str(y).strip().lower()
    if s in ["1","true","t","yes","y"]: return 1
    if s in ["0","false","f","no","n"]: return 0
    try: return int(float(s))
    except: return 0


# ----------------------------
# Focus code (label-aware selection, but label-blind prompting)
# ----------------------------
def get_focus_code(row: pd.Series, schema: DatasetSchema) -> str:
    y = _to_int_label(row.get(schema.label_col, 0))
    if y == 1 and schema.before_code_col:
        v = _clean_code(row.get(schema.before_code_col, ""))
        if v:
            return v
    if y == 0 and schema.after_code_col:
        v = _clean_code(row.get(schema.after_code_col, ""))
        if v:
            return v
    return _clean_code(row.get(schema.code_col, ""))


# ----------------------------
# Chunking
# ----------------------------
def chunk_text(text: str, mode: str, max_chars: int = 1400, overlap: int = 200) -> List[str]:
    if mode == "code":
        text = _clean_code(text)
    else:
        text = _clean_nl(text)

    if not text:
        return []
    if len(text) <= max_chars:
        return [text]

    out = []
    start = 0
    while start < len(text):
        end = min(len(text), start + max_chars)
        out.append(text[start:end].strip())
        if end == len(text):
            break
        start = max(0, end - overlap)
    return [c for c in out if c]


def build_chunks_for_row(row: pd.Series, schema: DatasetSchema) -> List[Tuple[str,str]]:
    chunks: List[Tuple[str,str]] = []

    def add(tag: str, val: Any, mode: str):
        for c in chunk_text(val, mode=mode):
            chunks.append((tag, c))

    add("focus_code", get_focus_code(row, schema), mode="code")

    if schema.diff_col:
        add("diff", row.get(schema.diff_col, ""), mode="code")

    if schema.commit_msg_col:
        add("commit_msg", row.get(schema.commit_msg_col, ""), mode="nl")

    if schema.before_code_col:
        add("before_code", row.get(schema.before_code_col, ""), mode="code")

    if schema.after_code_col:
        add("after_code", row.get(schema.after_code_col, ""), mode="code")

    if schema.cve_desc_col:
        add("cve_desc", row.get(schema.cve_desc_col, ""), mode="nl")

    meta = []
    for col, tag in [
        (schema.cve_id_col, "cve_id"),
        (schema.cwe_col, "cwe"),
        (schema.repo_col, "repo"),
        (schema.file_path_col, "file"),
        (schema.func_name_col, "func"),
        (schema.url_col, "url"),
    ]:
        if col:
            v = _clean_nl(row.get(col, ""))
            if v:
                meta.append(f"{tag}: {v}")
    if meta:
        chunks.append(("meta", " | ".join(meta)))

    return [(t, x) for t, x in chunks if x]


# ----------------------------
# Label-blind query
# ----------------------------
def make_query(row: pd.Series, schema: DatasetSchema) -> str:
    cve = _clean_nl(row.get(schema.cve_id_col, "")) if schema.cve_id_col else ""
    cwe = _clean_nl(row.get(schema.cwe_col, "")) if schema.cwe_col else ""
    fn  = _clean_nl(row.get(schema.func_name_col, "")) if schema.func_name_col else ""
    repo = _clean_nl(row.get(schema.repo_col, "")) if schema.repo_col else ""
    fp   = _clean_nl(row.get(schema.file_path_col, "")) if schema.file_path_col else ""
    msg  = _clean_nl(row.get(schema.commit_msg_col, "")) if schema.commit_msg_col else ""
    q = f"{cve} {cwe} {fn} {repo} {fp} {msg}".strip()
    q = re.sub(r"\s+", " ", q).strip()
    return (q + " security vulnerability fix patch diff").strip() if q else "security vulnerability fix patch diff"


# ----------------------------
# Retriever
# ----------------------------
class RowLocalRetriever:
    def __init__(self, emb_model_name=EMB_MODEL, rerank_model_name=RERANK_MODEL):
        self.emb = SentenceTransformer(emb_model_name)
        self.reranker = CrossEncoder(rerank_model_name) if rerank_model_name else None

    def retrieve(
        self,
        row: pd.Series,
        schema: DatasetSchema,
        k=TOPK,
        force_types=("focus_code","diff","commit_msg","before_code","after_code","meta","cve_desc"),
    ):
        chunks = build_chunks_for_row(row, schema)
        if not chunks:
            return []

        qtxt = make_query(row, schema)

        q = self.emb.encode([f"query: {qtxt}"], normalize_embeddings=True)
        X = self.emb.encode([f"passage: {c[1]}" for c in chunks], normalize_embeddings=True)

        sims = (X @ q[0]).astype(float)
        order = np.argsort(-sims)

        cand = []
        for i in order[:min(len(order), max(k * 3, 24))]:
            cand.append({"doc_id": int(i), "chunk_type": chunks[i][0], "text": chunks[i][1], "score": float(sims[i])})

        if self.reranker is not None and len(cand) > 1:
            pairs = [(qtxt, r["text"]) for r in cand[:min(len(cand), 24)]]
            rr = self.reranker.predict(pairs)
            for r, s in zip(cand[:len(pairs)], rr):
                r["rerank_score"] = float(s)
            cand = sorted(cand, key=lambda r: r.get("rerank_score", r["score"]), reverse=True)

        def key(r): return r.get("rerank_score", r["score"])

        best = {}
        for r in cand:
            t = r["chunk_type"]
            if (t not in best) or (key(r) > key(best[t])):
                best[t] = r

        out, used = [], set()
        for t in force_types:
            if t in best:
                out.append(best[t])
                used.add((best[t]["chunk_type"], best[t]["doc_id"]))

        for r in cand:
            kk = (r["chunk_type"], r["doc_id"])
            if kk not in used:
                out.append(r)
                used.add(kk)
            if len(out) >= k:
                break

        return out[:k]


# ----------------------------
# LLM JSON validation
# ----------------------------
def validate_json(js: dict, retrieved: List[Dict[str, Any]]) -> bool:
    if not isinstance(js, dict):
        return False

    if "predicted_label" not in js or "pred_confidence" not in js:
        return False

    try:
        pl = int(js.get("predicted_label"))
    except:
        return False
    if pl not in [0, 1]:
        return False

    try:
        pc = float(js.get("pred_confidence"))
    except:
        return False
    if pc < 0 or pc > 1:
        return False

    ke = js.get("key_evidence", None)
    if not isinstance(ke, list) or len(ke) == 0:
        return False

    doc_ids = {int(r["doc_id"]) for r in retrieved}
    ok = False
    for ev in ke[:3]:
        if isinstance(ev, dict) and "doc_id" in ev and "snippet" in ev:
            try:
                did = int(ev["doc_id"])
            except:
                continue
            sn = str(ev["snippet"]).strip()
            if did in doc_ids and len(sn) >= 10:
                ok = True
                break
    return ok


class VerdictLLM:
    def __init__(self, model_name=MODEL_NAME):
        # CRITICAL FIX: DO NOT use trust_remote_code (prevents outdated modeling_phi3.py) :contentReference[oaicite:2]{index=2}
        self.tok = AutoTokenizer.from_pretrained(model_name, use_fast=True, trust_remote_code=False)
        self.tok.truncation_side = "left"

        dtype = torch.float16 if torch.cuda.is_available() else torch.float32
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=dtype,
            trust_remote_code=False,
            device_map="auto" if torch.cuda.is_available() else None,
        )

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.eval()

        if self.tok.pad_token_id is None:
            self.tok.pad_token = self.tok.eos_token

    def _format_chat(self, user_text: str) -> str:
        if hasattr(self.tok, "apply_chat_template"):
            msgs = [
                {"role": "system", "content": "Return ONLY valid JSON. No extra text."},
                {"role": "user", "content": user_text},
            ]
            return self.tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        return user_text

    def _extract_json(self, text: str) -> Optional[dict]:
        text = text.strip()
        text = re.sub(r"^```(?:json)?\s*", "", text)
        text = re.sub(r"\s*```$", "", text)

        m = re.search(r"\{[\s\S]*\}", text)
        if not m:
            return None
        cand = m.group(0).strip()
        cand = re.sub(r",\s*}", "}", cand)
        cand = re.sub(r",\s*]", "]", cand)
        try:
            return json.loads(cand)
        except:
            return None

    def generate(self, user_prompt: str, retrieved: List[Dict[str, Any]]):
        p = user_prompt
        for _ in range(RETRIES + 1):
            prompt = self._format_chat(p)
            inputs = self.tok(prompt, return_tensors="pt", truncation=True, max_length=MAX_PROMPT_LEN).to(self.device)

            with torch.no_grad():
                out = self.model.generate(
                    **inputs,
                    max_new_tokens=MAX_NEW_TOKENS,
                    do_sample=False,
                    temperature=0.0,
                    top_p=1.0,
                    eos_token_id=self.tok.eos_token_id,
                    pad_token_id=self.tok.pad_token_id,

                    # EXTRA SAFETY: disable cache (avoids cache API mismatches)
                    use_cache=False,
                )

            prompt_len = inputs["input_ids"].shape[1]
            gen_ids = out[0][prompt_len:]
            text = self.tok.decode(gen_ids, skip_special_tokens=True).strip()

            js = self._extract_json(text)
            if js is not None and validate_json(js, retrieved):
                return js

            p = p + "\n\nIMPORTANT: Output ONLY JSON. Start with { and end with }. No extra text."
        fallback_doc = retrieved[0]["doc_id"] if retrieved else 0
        return {
            "predicted_label": 0,
            "pred_confidence": 0.0,
            "rationale": "Invalid JSON output.",
            "key_evidence": [{"doc_id": fallback_doc, "snippet": "(no valid json)"}],
        }


def build_prompt(row: pd.Series, schema: DatasetSchema, retrieved: List[Dict[str, Any]]) -> str:
    ctx = []
    for r in retrieved[:6]:
        ctx.append(f"[doc_id={r['doc_id']} | {r['chunk_type']}]\n{r['text'][:1000]}\n")
    ctx = "\n---\n".join(ctx)

    return f"""You are a careful security analyst.

Primary instruction:
- Treat 'focus_code' as the main code to classify.
- Use diff/commit_msg/before_code/after_code/meta only as supporting context.

Decide whether the focus_code is:
- predicted_label = 1  (vulnerable / pre-fix unsafe)
- predicted_label = 0  (fixed / mitigated / not vulnerable)

Confidence calibration (do NOT always use 0.90):
- weak/ambiguous evidence -> pred_confidence <= 0.60
- partial hints -> 0.60 to 0.80
- explicit vulnerability OR explicit mitigation -> 0.80 to 0.95

Evidence:
{ctx}

Return ONLY JSON exactly:
{{
  "predicted_label": 0 or 1,
  "pred_confidence": 0 to 1,
  "rationale": "1 short sentence",
  "key_evidence": [{{"doc_id": 0, "snippet": "exact copy from Evidence"}}]
}}
"""


def compute_verdict(dataset_label: int, predicted_label: int, pred_conf: float) -> str:
    if pred_conf < 0.60:
        return "UNCERTAIN"
    return "VERIFIED" if predicted_label == dataset_label else "CONTRADICTED"


def score_full(df: pd.DataFrame, schema: DatasetSchema):
    done_ids = set()
    rows_out = []

    print("FORCE_FRESH_RUN:", FORCE_FRESH_RUN)
    print("OUT_PART:", OUT_PART)
    print("OUT_FULL:", OUT_FULL)

    if (not FORCE_FRESH_RUN) and os.path.exists(OUT_PART):
        part = pd.read_csv(OUT_PART)
        if schema.id_col in part.columns:
            done_ids = set(part[schema.id_col].astype(int).tolist())
            rows_out = part.to_dict("records")
            print(f"[RESUME] Loaded {len(done_ids)} done rows from {OUT_PART}.")

    df_run = df[~df[schema.id_col].astype(int).isin(done_ids)].copy().reset_index(drop=True)
    print("To score:", len(df_run), "/", len(df))

    retriever = RowLocalRetriever()
    llm = VerdictLLM()

    for i, row in df_run.iterrows():
        retrieved = retriever.retrieve(row, schema, k=TOPK)
        prompt = build_prompt(row, schema, retrieved)
        js = llm.generate(prompt, retrieved)

        dataset_y = _to_int_label(row.get(schema.label_col, 0))
        predicted_label = int(js.get("predicted_label", 0))
        pred_conf = float(js.get("pred_confidence", 0.0) or 0.0)

        verdict = compute_verdict(dataset_y, predicted_label, pred_conf)

        rows_out.append({
            schema.id_col: int(row[schema.id_col]),
            "dataset_label": int(dataset_y),
            "predicted_label": int(predicted_label),
            "pred_confidence": float(pred_conf),
            "verdict": verdict,
            "confidence": float(pred_conf),
            "validity_score": float(pred_conf),
            "rationale": js.get("rationale", ""),
            "key_evidence": json.dumps(js.get("key_evidence", []), ensure_ascii=False),
            "retrieved_types": ",".join([r["chunk_type"] for r in retrieved]) if retrieved else "",
        })

        if (len(rows_out) % SAVE_EVERY) == 0:
            pd.DataFrame(rows_out).to_csv(OUT_PART, index=False)
            print(f"[checkpoint] saved {len(rows_out)} rows -> {OUT_PART}")

    pd.DataFrame(rows_out).to_csv(OUT_PART, index=False)

    score_df = pd.DataFrame(rows_out)
    final = df.merge(score_df, on=schema.id_col, how="left")
    final.to_csv(OUT_FULL, index=False)

    print("[saved]", OUT_FULL)
    print(final["verdict"].value_counts(dropna=False))
    print(final["confidence"].describe())

    # Diagnostics: mismatches
    mismatch = (final["dataset_label"].astype(int) != final["predicted_label"].astype(int))
    print("\n[DIAGNOSTICS]")
    print("Mismatch count:", int(mismatch.sum()))
    print("Mismatch rate:", float(mismatch.mean()))
    print(pd.crosstab(final["dataset_label"], final["predicted_label"], rownames=["dataset_label"], colnames=["predicted_label"]))

    if mismatch.sum() > 0:
        cols = [schema.id_col, "dataset_label", "predicted_label", "pred_confidence", "verdict", "rationale"]
        print("\nExamples of mismatches (first 10):")
        print(final.loc[mismatch, cols].head(10).to_string(index=False))
    else:
        print("\nNOTE: mismatch=0 => CONTRADICTED legitimately 0 (model agrees with labels).")

    return final


# ==========================================================
# RUN FULL 2433 ON MEGAVUL
# ==========================================================
df = pd.read_csv("megavul_rich_2433.csv")

if "sample_id" not in df.columns:
    df["sample_id"] = np.arange(len(df))

df["diff_all"] = df.get("diff_func", "").astype(str) + "\n" + df.get("diff_line_info", "").astype(str)

df["is_vul"] = df["is_vul"].apply(_to_int_label)

schema = DatasetSchema(
    id_col="sample_id",
    code_col="func",
    label_col="is_vul",
    commit_msg_col="commit_msg",
    diff_col="diff_all",
    before_code_col="func_before",
    after_code_col="func",
    cve_id_col="cve_id",
    cwe_col="cwe_ids",
    repo_col="repo_name",
    url_col="git_url",
    file_path_col="file_path",
    func_name_col="func_name",
    cve_desc_col=None
)

final = score_full(df, schema)


FORCE_FRESH_RUN: True
OUT_PART: megavul_2433_scored_llm_rag_partial_20260204_133923.csv
OUT_FULL: megavul_2433_scored_llm_rag_FULL_20260204_133923.csv
To score: 2433 / 2433


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[checkpoint] saved 50 rows -> megavul_2433_scored_llm_rag_partial_20260204_133923.csv
[checkpoint] saved 100 rows -> megavul_2433_scored_llm_rag_partial_20260204_133923.csv
[checkpoint] saved 150 rows -> megavul_2433_scored_llm_rag_partial_20260204_133923.csv
[checkpoint] saved 200 rows -> megavul_2433_scored_llm_rag_partial_20260204_133923.csv



KeyboardInterrupt

